# 01 — CVE Severity & CVSS Prediction

**Status: research notebook (EXPERIMENTAL).** Nothing here is wired into the OffensiveX application, and no artifact produced by this notebook is production-ready merely because training completed. See `notebooks/README.md` and `docs/roadmap/ml-integration-contract.md`.

## Purpose
Investigate whether machine learning can predict **vulnerability severity** from CVE metadata, and — just as important — *measure how much of any apparent skill is merely re-deriving the official CVSS formula*.

## Research questions
1. **RQ1 (Experiment A — reconstruction benchmark).** Given the official CVSS *base metrics* (AV, AC, PR, UI, S, C, I, A…), how closely can an ML model reproduce the official base score / severity band? *This is NOT evidence of novel predictive capability* — CVSS is a deterministic function of those metrics, and an exact calculator exists (implemented below and validated against every score in the dataset).
2. **RQ2 (Experiment B — metadata-based prediction).** Using only information that exists when a CVE record is first published (description text, reporting CNA, CWE, text-derived flags), how well can a model predict the CVSS base score and qualitative severity? How much does performance degrade on *later* CVEs (chronological split)?
3. **RQ3 (leakage sensitivity).** How much does performance change if post-publication enrichment fields (reference count, CPE-match count) are added?
4. **RQ4.** Do results differ by CVSS version, by scoring source (NVD vs CNA), by year, or by CWE?

## Scope decisions (read before interpreting results)
* CVSS **v2, v3.0, v3.1 and v4.0 are never mixed**. The primary target is **CVSS v3.1** (largest sample). Other versions appear only in the per-version reconstruction table, each evaluated separately.
* Labels are CVSS qualitative bands for that version (v3.x/v4.0: NONE 0.0, LOW 0.1–3.9, MEDIUM 4.0–6.9, HIGH 7.0–8.9, CRITICAL 9.0–10.0). They map to OffensiveX's `FindingSeverity` enum (`info/low/medium/high/critical`) via `CVSS_TO_OFFENSIVEX_SEVERITY`.
* OffensiveX's authoritative risk number remains the deterministic `risk_model_v1` (`docs/security/risk-analysis.md`). This notebook does not replace or modify it. See ADR-009 in `docs/Decision.md`: an ML severity model that merely reproduces public scores "adds no information" — Experiment A quantifies exactly that.

## 2. Environment and dependency setup
Nothing is installed silently. On Kaggle/Colab all required packages are preinstalled. Locally: `pip install numpy pandas scikit-learn matplotlib seaborn joblib pyarrow` (`pyarrow` only for `.parquet` inputs). `shap` is **not** required.

In [ ]:
import os, sys, re, io, gc, json, gzip, math, time, hashlib, platform, warnings, random, importlib.util
import urllib.request, urllib.error
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import joblib

try:
    from IPython.display import display, Markdown
except ImportError:  # plain-python fallback so the code also runs as a script
    display, Markdown = print, str

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 80)

# ---- environment detection (used ONLY for paths/setup, never to change the methodology) ----
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
ENVIRONMENT = "kaggle" if IN_KAGGLE else "colab" if IN_COLAB else "local"

# ---- dependency check (nothing is installed silently) ----
_REQUIRED = ["numpy", "pandas", "sklearn", "matplotlib", "seaborn", "joblib"]
_OPTIONAL = {"pyarrow": "needed only to read .parquet inputs", "shap": "optional; permutation importance is used if absent"}
_missing = [m for m in _REQUIRED if importlib.util.find_spec(m) is None]
if _missing:
    raise ImportError(
        f"Missing required packages: {_missing}. Install with: pip install numpy pandas scikit-learn matplotlib seaborn joblib pyarrow"
    )
OPTIONAL_AVAILABLE = {m: importlib.util.find_spec(m) is not None for m in _OPTIONAL}


def _dependency_versions():
    out = {"python": sys.version.split()[0], "platform": platform.platform()}
    for name in ["numpy", "pandas", "sklearn", "scipy", "matplotlib", "seaborn", "joblib", "pyarrow", "shap"]:
        try:
            out[name] = importlib.import_module(name).__version__
        except Exception:
            out[name] = None
    return out


DEPENDENCY_VERSIONS = _dependency_versions()


def resolve_default_dirs(cfg):
    """Fill OUTPUT_ROOT / CACHE_DIR when left as None. Override anytime via CONFIG or the OFFENSIVEX_ML_OUTPUT env var."""
    if cfg.get("OUTPUT_ROOT") is None:
        if os.environ.get("OFFENSIVEX_ML_OUTPUT"):
            cfg["OUTPUT_ROOT"] = os.environ["OFFENSIVEX_ML_OUTPUT"]
        elif IN_KAGGLE:
            cfg["OUTPUT_ROOT"] = "/kaggle/working/artifacts/ml-research"
        elif IN_COLAB:
            cfg["OUTPUT_ROOT"] = "/content/artifacts/ml-research"
        else:  # local Jupyter: <repo>/artifacts/ml-research (git-ignored), whether launched from the repo root or notebooks/
            base = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
            cfg["OUTPUT_ROOT"] = str(base / "artifacts" / "ml-research")
    if cfg.get("CACHE_DIR") is None:
        cfg["CACHE_DIR"] = str(Path(cfg["OUTPUT_ROOT"]).parent / "data_cache")
    return cfg


def set_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

## 3. Dataset sources, access instructions, licensing and limitations

| Source | Identifier / URL (verified 2026-10-14) | Access | License / terms | Used for |
|---|---|---|---|---|
| **Kaggle: "NVD CVE CVSS KEV EPSS Vulnerabilities 2020-2026"** | `afr1ste/nvd-cve-cvss-kev-vulnerabilities-2020-2026`, file `nvd_cve_records.parquet` (41.8 MB; CSV 181.6 MB) — 201,542 CVEs, 106 columns, snapshot built 2026-05-02 | **Attach via Kaggle Input panel** (no download needed); Colab/local: Kaggle API/CLI with your own token | CC0 (per Kaggle listing). Underlying NVD data is US-government data | Primary input (offline-capable) |
| **NVD CVE JSON 2.0 yearly feeds** | `https://nvd.nist.gov/feeds/json/cve/2.0/nvdcve-2.0-<year>.json.gz` (10–25 MB/year) | Public HTTPS, no key. Used only if `ALLOW_NETWORK=True` and nothing is attached | NVD data is public; see <https://nvd.nist.gov/general/legal-disclaimer> | Alternative primary input |
| NVD CVE API 2.0 | `https://services.nvd.nist.gov/rest/json/cves/2.0` | **Not used** — bulk feeds are cheaper (the API is paginated at ≤2000/page and rate-limited to 5 req/30 s without a key) | — | — |

**Dataset facts verified before use:** the Kaggle table carries per-version columns `cvss_{v40,v31,v30,v2}_{vector,base_score,base_severity,source,type,...}`; **the CVSS v2 component columns are empty in that table**, so this notebook parses metrics from the *vector strings* for every version (and cross-checks scores against official calculators). Yearly NVD feeds are keyed by **CVE-ID year** (a `nvdcve-2.0-2023` file contains CVEs published as late as 2026) — hence the dataset omits e.g. `CVE-2019-*` records published in 2020+.

**Known limitations / biases**
* Scores come from heterogeneous *scorers* (NVD analysts vs. CNAs, `cvss_*_type/source`), with different habits. NVD has not scored CVSS v2 for new CVEs since mid-2022 and v4.0 is still sparse (≈11% of rows).
* A snapshot reflects **current** NVD state (last-modified up to the snapshot), not the record as first published. Scores/CWEs may have been added or changed after publication.
* Severity classes are imbalanced (CRITICAL/HIGH/MEDIUM ≫ LOW; NONE ≈ 0).
* Descriptions are written by CNAs and frequently contain the very words used to assign CVSS metrics ("remote attacker", "unauthenticated") — informative but partly circular with the scorer's reasoning.

**Access from each environment**
* **Kaggle:** Notebook → *Input* → *Add Input* → search the slug above. Internet can stay **off**. Files appear under `/kaggle/input/<dataset-folder>/` and are read in place (nothing is copied).
* **Colab:** download once with the Kaggle API (token in *Secrets* as `KAGGLE_USERNAME` / `KAGGLE_KEY`; never paste tokens into cells), e.g. `kaggle datasets download -d afr1ste/nvd-cve-cvss-kev-vulnerabilities-2020-2026 -f nvd_cve_records.parquet`, unzip, set `DATASET_PATH`. Or set `ALLOW_NETWORK=True` to fetch the official feeds.
* **Local:** put the parquet (or feeds) somewhere and set `DATASET_PATH`, or set `ALLOW_NETWORK=True`.

## 4. Configuration variables
**What you may need to change:** `DATASET_PATH` (only if auto-discovery fails), `ALLOW_NETWORK` (only for the official-feed route), `NVD_YEARS` (feed route only), `OUTPUT_ROOT` (optional). Everything else is methodology: change it deliberately and re-run all cells top to bottom.
`FAST_DEV=True` subsamples rows for debugging — artifacts from such a run are flagged `fast_dev=true` and must not be reported.

In [ ]:
CONFIG = dict(
    # ---- environment/data (change if needed) ----
    DATASET_PATH=None,          # None = auto-discover; or a folder/file with nvd_cve_records.parquet|csv, or nvdcve-2.0-<year>.json.gz
    ALLOW_NETWORK=False,        # True = may download official NVD feeds into CACHE_DIR when no local data is found
    NVD_YEARS=list(range(2020, 2027)),   # CVE-ID years for the official-feed route
    OUTPUT_ROOT=None,           # None = auto (<repo>/artifacts/ml-research locally, /kaggle/working/... on Kaggle)
    CACHE_DIR=None,             # None = auto
    EXPERIMENT_DIR="severity",
    FAST_DEV=False,             # debugging only: random row subsample, artifacts flagged non-reportable
    FAST_DEV_ROWS=40_000,
    # ---- methodology (documented defaults) ----
    SEED=42,
    TARGET_VERSION="v31",       # one of v31, v30, v2, v40 -- versions are NEVER mixed
    MIN_ROWS_PER_VERSION=2_000, # minimum rows for a per-version reconstruction run
    MIN_CLASS_ROWS=100,         # classes with fewer rows are excluded from classification (documented)
    SPLIT_FRACTIONS=(0.70, 0.15, 0.15),   # chronological train / validation / test
    N_REPEATED_SPLITS=5,        # repeated random splits for variability
    N_BOOTSTRAP=500,            # bootstrap resamples for test-metric CIs
    TFIDF_MAX_FEATURES=20_000,
    SVD_COMPONENTS=64,          # dimensionality of the text representation fed to tree models
    PERM_IMPORTANCE_ROWS=5_000,
    PLOT_SAMPLE_ROWS=20_000,    # exploratory plots only; training always uses all rows
    N_JOBS=-1,
)
resolve_default_dirs(CONFIG)
set_seeds(CONFIG["SEED"])

In [ ]:
# ======================= CVSS helpers: vector parsing, official calculators, severity bands =======================
# Formulas: FIRST CVSS v3.1 specification section 7 (https://www.first.org/cvss/v3-1/specification-document),
# v3.0 (https://www.first.org/cvss/v3.0/specification-document), v2 (https://www.first.org/cvss/v2/guide).
# CVSS v4.0 scoring uses a macro-vector lookup table and is deliberately NOT re-implemented here.
CVSS_VERSIONS = {"v40": "4.0", "v31": "3.1", "v30": "3.0", "v2": "2.0"}


def parse_cvss_vector(vec):
    """'CVSS:3.1/AV:N/...' or (v2) 'AV:N/AC:L/...' -> (version, {metric: value}); None if unparsable."""
    if not isinstance(vec, str) or not vec:
        return None
    version, metrics = None, {}
    for part in vec.split("/"):
        if ":" not in part:
            return None
        k, v = part.split(":", 1)
        if k == "CVSS":
            version = v
        else:
            metrics[k] = v
    return (version or "2.0"), metrics


_W31 = {"AV": {"N": 0.85, "A": 0.62, "L": 0.55, "P": 0.2}, "AC": {"L": 0.77, "H": 0.44},
        "UI": {"N": 0.85, "R": 0.62}, "CIA": {"H": 0.56, "L": 0.22, "N": 0.0},
        "PR_U": {"N": 0.85, "L": 0.62, "H": 0.27}, "PR_C": {"N": 0.85, "L": 0.68, "H": 0.5}}


def _roundup_31(x):
    i = int(round(x * 100000))
    return i / 100000.0 if i % 10000 == 0 else (math.floor(i / 10000) + 1) / 10.0


def cvss3_base_score(vector):
    """Official CVSS v3.0/v3.1 base score from a vector string (None if the vector is invalid)."""
    p = parse_cvss_vector(vector)
    if p is None or p[0] not in ("3.0", "3.1"):
        return None
    ver, m = p
    try:
        changed = m["S"] == "C"
        iss = 1 - (1 - _W31["CIA"][m["C"]]) * (1 - _W31["CIA"][m["I"]]) * (1 - _W31["CIA"][m["A"]])
        impact = 7.52 * (iss - 0.029) - 3.25 * (iss - 0.02) ** 15 if changed else 6.42 * iss
        pr = (_W31["PR_C"] if changed else _W31["PR_U"])[m["PR"]]
        expl = 8.22 * _W31["AV"][m["AV"]] * _W31["AC"][m["AC"]] * pr * _W31["UI"][m["UI"]]
    except KeyError:
        return None
    if impact <= 0:
        return 0.0
    raw = min((impact + expl) * (1.08 if changed else 1.0), 10)
    return _roundup_31(raw) if ver == "3.1" else math.ceil(raw * 10 - 1e-9) / 10.0


def cvss2_base_score(vector):
    p = parse_cvss_vector(vector)
    if p is None or p[0] != "2.0":
        return None
    m = p[1]
    try:
        av, ac, au = {"L": 0.395, "A": 0.646, "N": 1.0}[m["AV"]], {"H": 0.35, "M": 0.61, "L": 0.71}[m["AC"]], {"M": 0.45, "S": 0.56, "N": 0.704}[m["Au"]]
        cia = {"N": 0.0, "P": 0.275, "C": 0.660}
        impact = 10.41 * (1 - (1 - cia[m["C"]]) * (1 - cia[m["I"]]) * (1 - cia[m["A"]]))
    except KeyError:
        return None
    expl = 20 * av * ac * au
    f = 0 if impact == 0 else 1.176
    return round(((0.6 * impact) + (0.4 * expl) - 1.5) * f + 1e-9, 1)


OFFICIAL_CALCULATORS = {"v31": cvss3_base_score, "v30": cvss3_base_score, "v2": cvss2_base_score}  # v40: none implemented


def cvss_severity(score, version):
    """Qualitative rating per the CVSS specification for the given version. Returns an upper-case label or None."""
    if score is None or (isinstance(score, float) and math.isnan(score)):
        return None
    if version == "v2":  # v2 has no NONE/CRITICAL band
        return "LOW" if score < 4.0 else "MEDIUM" if score < 7.0 else "HIGH"
    if score == 0.0:
        return "NONE"
    return "LOW" if score < 4.0 else "MEDIUM" if score < 7.0 else "HIGH" if score < 9.0 else "CRITICAL"


# Mapping to OffensiveX's finding severity enum (backend/src/offensivex/db/models.py::FindingSeverity).
CVSS_TO_OFFENSIVEX_SEVERITY = {"NONE": "info", "LOW": "low", "MEDIUM": "medium", "HIGH": "high", "CRITICAL": "critical"}
CVE_ID_RE = re.compile(r"^CVE-\d{4}-\d{4,}$")
assert CONFIG["TARGET_VERSION"] in CVSS_VERSIONS, f"TARGET_VERSION must be one of {list(CVSS_VERSIONS)}"
TV = CONFIG["TARGET_VERSION"]
TV_NAME = f"CVSS v{CVSS_VERSIONS[TV]}"

In [ ]:
# ======================= dataset discovery, loading, fingerprinting =======================
# Supported inputs (auto-detected, in this order):
#   1. A "canonical table" named nvd_cve_records.parquet / .csv  (Kaggle: afr1ste/nvd-cve-cvss-kev-vulnerabilities-2020-2026)
#   2. Official NVD CVE JSON 2.0 yearly feeds named nvdcve-2.0-<year>.json.gz found locally
#   3. Official NVD feeds downloaded into CACHE_DIR -- ONLY when CONFIG["ALLOW_NETWORK"] is True
# Nothing is downloaded when a local source is found, and nothing is copied out of /kaggle/input.

SUPPORTED_SUFFIXES = (".parquet", ".csv", ".csv.gz", ".json", ".json.gz", ".jsonl", ".jsonl.gz")
NVD_FEED_URL = "https://nvd.nist.gov/feeds/json/cve/2.0/nvdcve-2.0-{year}.json.gz"
KEV_URL = "https://www.cisa.gov/sites/default/files/csv/known_exploited_vulnerabilities.csv"
EPSS_URL = "https://epss.cyentia.com/epss_scores-current.csv.gz"
KEV_NAMES = ("known_exploited_vulnerabilities.csv", "cisa_kev_catalog.csv")
EPSS_NAMES = ("epss_scores-current.csv.gz", "first_epss_current.csv")
_SKIP_DIRS = {"node_modules", "__pycache__", "artifacts", ".venv", "venv", "site-packages", "sample_data"}


class DataNotFoundError(RuntimeError):
    pass


def _roots_to_scan():
    if CONFIG["DATASET_PATH"]:
        return [Path(CONFIG["DATASET_PATH"]).expanduser()]
    roots = []
    if IN_KAGGLE:
        roots.append(Path("/kaggle/input"))
    roots += [Path(CONFIG["CACHE_DIR"]), Path.cwd(), Path.cwd() / "data"]
    if IN_COLAB:
        roots.append(Path("/content"))
    seen, out = set(), []
    for r in roots:
        if r.exists() and str(r.resolve()) not in seen:
            seen.add(str(r.resolve()))
            out.append(r)
    return out


def list_input_files(roots, max_depth=4, limit=500):
    found = []
    for root in roots:
        if root.is_file():
            found.append(root)
            continue
        base = len(root.resolve().parts)
        for dirpath, dirnames, filenames in os.walk(root):
            if len(Path(dirpath).resolve().parts) - base >= max_depth:
                dirnames[:] = []
            dirnames[:] = [d for d in dirnames if not d.startswith(".") and d not in _SKIP_DIRS]
            for fn in filenames:
                if fn.lower().endswith(SUPPORTED_SUFFIXES):
                    found.append(Path(dirpath) / fn)
            if len(found) >= limit:
                return found
    return found


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()


def http_download(url, dest, tries=4, timeout=120, backoff=5):
    """Cached, retrying, rate-limit-aware download (honours Retry-After on 429/503). Atomic write."""
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"  cache hit: {dest.name} ({dest.stat().st_size/1e6:.1f} MB) -- delete it to refresh")
        return dest
    part = Path(str(dest) + ".part")
    last, wait = None, backoff
    for attempt in range(1, tries + 1):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "OffensiveX-ml-research/1.0 (research notebook)"})
            with urllib.request.urlopen(req, timeout=timeout) as r, open(part, "wb") as f:
                while block := r.read(1 << 20):
                    f.write(block)
            part.replace(dest)
            print(f"  downloaded {url} -> {dest.name} ({dest.stat().st_size/1e6:.1f} MB)")
            return dest
        except urllib.error.HTTPError as e:
            last = e
            if e.code in (400, 401, 403, 404):
                raise RuntimeError(f"HTTP {e.code} for {url} (not retryable)") from e
            ra = e.headers.get("Retry-After") if e.headers else None
            wait = int(ra) if ra and ra.isdigit() else backoff * attempt * (2 if e.code in (429, 503) else 1)
        except (urllib.error.URLError, TimeoutError, ConnectionError) as e:
            last, wait = e, backoff * attempt
        print(f"  attempt {attempt}/{tries} failed for {url}: {last}; retrying in {wait}s")
        time.sleep(wait)
    raise RuntimeError(f"Could not download {url} after {tries} attempts: {last}")


# ----------------------------- NVD CVE JSON 2.0 parsing -----------------------------
_CVSS_KEYS = {"v40": "cvssMetricV40", "v31": "cvssMetricV31", "v30": "cvssMetricV30", "v2": "cvssMetricV2"}
_CVSS_FIELDS = ["vector", "base_score", "base_severity", "source", "type", "exploitability_score", "impact_score"]
BASE_COLUMNS = ["cve_id", "source_identifier", "published_date", "last_modified_date", "vuln_status", "description_en",
                "reference_count", "weakness_count", "cpe_match_count", "primary_cwe_id"]
CANON_COLUMNS = BASE_COLUMNS + [f"cvss_{v}_{f}" for v in _CVSS_KEYS for f in _CVSS_FIELDS]


def _count_cpe_matches(configs):
    n = 0
    for conf in configs or []:
        stack = list(conf.get("nodes", []))
        while stack:
            node = stack.pop()
            n += len(node.get("cpeMatch", []))
            stack.extend(node.get("children", []))
    return n


def parse_nvd_record(rec):
    c = rec["cve"]
    row = {"cve_id": c.get("id"), "source_identifier": c.get("sourceIdentifier"),
           "published_date": c.get("published"), "last_modified_date": c.get("lastModified"),
           "vuln_status": c.get("vulnStatus")}
    row["description_en"] = next((d["value"] for d in c.get("descriptions", []) if d.get("lang") == "en"), None)
    row["reference_count"] = len(c.get("references", []))
    cwes = [d["value"] for w in c.get("weaknesses", []) for d in w.get("description", [])]
    row["weakness_count"] = len(set(cwes))
    row["primary_cwe_id"] = cwes[0] if cwes else None
    row["cpe_match_count"] = _count_cpe_matches(c.get("configurations"))
    metrics = c.get("metrics", {})
    for ver, key in _CVSS_KEYS.items():
        entries = metrics.get(key) or []
        m = next((e for e in entries if e.get("type") == "Primary"), entries[0] if entries else None)
        p = f"cvss_{ver}_"
        if m is None:
            continue
        d = m.get("cvssData", {})
        row[p + "vector"] = d.get("vectorString")
        row[p + "base_score"] = d.get("baseScore")
        row[p + "base_severity"] = d.get("baseSeverity") or m.get("baseSeverity")
        row[p + "source"] = m.get("source")
        row[p + "type"] = m.get("type")
        row[p + "exploitability_score"] = m.get("exploitabilityScore")
        row[p + "impact_score"] = m.get("impactScore")
    return row


def parse_nvd_feed(path):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        data = json.load(f)
    if data.get("format") != "NVD_CVE" or "vulnerabilities" not in data:
        raise ValueError(f"{path} is not an NVD CVE 2.0 feed (format={data.get('format')!r})")
    return pd.DataFrame([parse_nvd_record(r) for r in data["vulnerabilities"]])


def read_kev(path):
    kev = pd.read_csv(path)
    low = {c.lower(): c for c in kev.columns}
    id_col = low.get("cveid") or low.get("cve_id")
    add_col = low.get("dateadded") or low.get("date_added")
    if id_col is None or add_col is None:
        raise ValueError(f"KEV file {path} lacks cveID/dateAdded columns: {list(kev.columns)}")
    out = kev[[id_col, add_col]].rename(columns={id_col: "cve_id", add_col: "kev_dateadded"})
    out["kev_dateadded"] = pd.to_datetime(out["kev_dateadded"], errors="coerce")
    return out.drop_duplicates("cve_id")


def read_epss(path):
    opener = gzip.open if str(path).endswith(".gz") else open
    with opener(path, "rt", encoding="utf-8") as f:
        header = f.readline()
    skip = 1 if header.startswith("#") else 0
    e = pd.read_csv(path, skiprows=skip)
    e.columns = [c.lower() for c in e.columns]
    if "cve" in e.columns:
        e = e.rename(columns={"cve": "cve_id"})
    meta = dict(re.findall(r"(\w+):([^,\s]+)", header)) if skip else {}
    e["epss_score_date"] = meta.get("score_date")
    e["epss_model_version"] = meta.get("model_version")
    return e.rename(columns={"epss": "epss_score", "percentile": "epss_percentile"})


def _find_named(files, names):
    lowered = {f.name.lower(): f for f in files}
    for n in names:
        if n in lowered:
            return lowered[n]
    return None


def _standardise(df):
    df = df.copy()
    for c in ("published_date", "last_modified_date"):
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce", utc=True).dt.tz_localize(None).astype("datetime64[ns]")
    if "kev_dateadded" in df.columns:
        df["kev_dateadded"] = pd.to_datetime(df["kev_dateadded"], errors="coerce")
    return df


def _read_table(path):
    p = str(path).lower()
    if p.endswith(".parquet"):
        return pd.read_parquet(path)
    if p.endswith((".csv", ".csv.gz")):
        return pd.read_csv(path, low_memory=False)
    raise ValueError(f"Unsupported file type for a canonical table: {path}")


def load_dataset(need_exploit_signals=False):
    """Returns (DataFrame in canonical schema, metadata dict). Raises DataNotFoundError with instructions."""
    roots = _roots_to_scan()
    print(f"Scanning for inputs under: {[str(r) for r in roots]}")
    files = list_input_files(roots)
    print(f"Found {len(files)} candidate data file(s):")
    for f in files[:25]:
        print(f"  {f}  ({f.stat().st_size/1e6:.1f} MB)")
    if len(files) > 25:
        print(f"  ... and {len(files) - 25} more")

    kind, sources = None, []
    canon = _find_named(files, ("nvd_cve_records.parquet", "nvd_cve_records.csv"))
    explicit = Path(CONFIG["DATASET_PATH"]).expanduser() if CONFIG["DATASET_PATH"] else None
    if canon is not None:
        kind, sources = "canonical_table", [canon]
    elif explicit is not None and explicit.is_file():
        kind, sources = "canonical_table", [explicit]
    else:
        feeds = sorted(f for f in files if (m := re.fullmatch(r"nvdcve-2\.0-(\d{4})\.json\.gz", f.name))
                       and int(m.group(1)) in set(CONFIG["NVD_YEARS"]))
        if feeds:
            kind, sources = "nvd_feeds", feeds
    if kind is None:
        if not CONFIG["ALLOW_NETWORK"]:
            raise DataNotFoundError(
                "No supported dataset found and CONFIG['ALLOW_NETWORK'] is False, so nothing will be downloaded.\n"
                "Do ONE of the following:\n"
                "  Kaggle : add the dataset 'afr1ste/nvd-cve-cvss-kev-vulnerabilities-2020-2026' via the notebook Input panel.\n"
                "  Any    : set CONFIG['DATASET_PATH'] to a folder/file containing nvd_cve_records.parquet|csv or nvdcve-2.0-<year>.json.gz.\n"
                "  Any    : set CONFIG['ALLOW_NETWORK']=True to download the official NVD yearly feeds (+KEV/EPSS) into CACHE_DIR."
            )
        print("No local dataset found; ALLOW_NETWORK=True -> downloading official NVD feeds into the cache directory.")
        sources = [http_download(NVD_FEED_URL.format(year=y), Path(CONFIG["CACHE_DIR"]) / f"nvdcve-2.0-{y}.json.gz")
                   for y in CONFIG["NVD_YEARS"]]
        kind = "nvd_feeds"
        files = files + list(sources)

    extra = []
    if kind == "nvd_feeds":
        parts = []
        for s in sources:
            t0 = time.time()
            parts.append(parse_nvd_feed(s))
            print(f"  parsed {Path(s).name}: {len(parts[-1]):,} CVEs in {time.time()-t0:.0f}s")
        df = pd.concat(parts, ignore_index=True)
        for col in CANON_COLUMNS:
            if col not in df.columns:
                df[col] = np.nan
        df = df.drop_duplicates("cve_id", keep="last")
        if need_exploit_signals:
            kev_f, epss_f = _find_named(files, KEV_NAMES), _find_named(files, EPSS_NAMES)
            if kev_f is None or epss_f is None:
                if not CONFIG["ALLOW_NETWORK"]:
                    raise DataNotFoundError(
                        "Exploit-signal files not found locally (need CISA KEV csv and FIRST EPSS csv.gz) and ALLOW_NETWORK is False.\n"
                        f"  KEV : {KEV_URL}\n  EPSS: {EPSS_URL}\nPlace them in CACHE_DIR or enable ALLOW_NETWORK."
                    )
                kev_f = kev_f or http_download(KEV_URL, Path(CONFIG["CACHE_DIR"]) / "known_exploited_vulnerabilities.csv")
                epss_f = epss_f or http_download(EPSS_URL, Path(CONFIG["CACHE_DIR"]) / "epss_scores-current.csv.gz")
            kev, epss = read_kev(kev_f), read_epss(epss_f)
            df = df.merge(kev, on="cve_id", how="left").merge(epss, on="cve_id", how="left")
            df["kev_is_known_exploited"] = df["kev_dateadded"].notna()
            extra = [kev_f, epss_f]
    else:
        df = _read_table(sources[0])
        if need_exploit_signals:
            for need in ("kev_is_known_exploited", "kev_dateadded", "epss_score"):
                if need not in df.columns:
                    raise DataNotFoundError(f"Canonical table lacks '{need}'; this notebook needs KEV/EPSS columns "
                                            "(use the afr1ste Kaggle dataset or the official-feed route with ALLOW_NETWORK).")
    df = _standardise(df)

    file_info = [{"name": Path(s).name, "bytes": Path(s).stat().st_size, "sha256": sha256_file(s)} for s in list(sources) + extra]
    fingerprint = hashlib.sha256(json.dumps([f["sha256"] for f in file_info] + [len(df)]).encode()).hexdigest()
    source_note = {
        "canonical_table": "Kaggle dataset afr1ste/nvd-cve-cvss-kev-vulnerabilities-2020-2026 (CC0 per Kaggle listing; derived from NVD/CISA KEV/FIRST EPSS)"
                           " or a user-supplied table with the same schema",
        "nvd_feeds": "Official NVD CVE JSON 2.0 yearly feeds (https://nvd.nist.gov/vuln/data-feeds) [+ CISA KEV + FIRST EPSS]",
    }[kind]
    meta = {"source_kind": kind, "source_description": source_note, "files": file_info, "dataset_fingerprint_sha256": fingerprint,
            "n_rows_loaded": int(len(df)), "n_columns": int(df.shape[1]), "environment": ENVIRONMENT,
            "loaded_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}
    return df, meta

In [ ]:
# ======================= output directories, plotting and artifact helpers =======================
OUTPUT_DIR = Path(CONFIG["OUTPUT_ROOT"]) / CONFIG["EXPERIMENT_DIR"]
PLOT_DIR = OUTPUT_DIR / "plots"
for _d in (OUTPUT_DIR, PLOT_DIR, Path(CONFIG["CACHE_DIR"])):
    _d.mkdir(parents=True, exist_ok=True)
SAVED_PLOTS = []
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 130, "axes.titlesize": 12, "axes.labelsize": 10})


def savefig(name, fig=None):
    """Save the figure to PLOT_DIR/<name>.png AND display it. Every plot in this notebook goes through here."""
    fig = fig or plt.gcf()
    fig.tight_layout()
    path = PLOT_DIR / f"{name}.png"
    fig.savefig(path, bbox_inches="tight")
    SAVED_PLOTS.append(path.name)
    plt.show()
    plt.close(fig)
    return path


def write_json(path, obj):
    def _default(o):
        if isinstance(o, np.integer):
            return int(o)
        if isinstance(o, np.floating):
            return None if np.isnan(o) else float(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        if isinstance(o, (pd.Timestamp, datetime)):
            return o.isoformat()
        return str(o)
    Path(path).write_text(json.dumps(obj, indent=2, default=_default), encoding="utf-8")
    return path


_SECRET_RE = re.compile(r"(?i)(api[_-]?key|secret|passw(?:or)?d|token|authorization)\"?\s*[:=]\s*\"?[A-Za-z0-9_\-/+=]{12,}")


def scan_for_secrets(directory):
    """Fail loudly if any exported text artifact looks like it contains a credential."""
    hits = []
    for p in Path(directory).rglob("*"):
        if p.is_file() and p.suffix.lower() in {".json", ".md", ".csv", ".txt"}:
            for m in _SECRET_RE.finditer(p.read_text(encoding="utf-8", errors="ignore")):
                hits.append((p.name, m.group(0)[:40]))
    if hits:
        raise RuntimeError(f"Possible secrets found in exported artifacts: {hits}")
    return True


def export_common(tables=None, extra_config=None):
    """Writes the metadata every experiment must ship: dataset metadata+fingerprint, dependency versions, config, exclusion ledger, tables."""
    write_json(OUTPUT_DIR / "dataset_metadata.json", DATA_META)
    write_json(OUTPUT_DIR / "dependency_versions.json", DEPENDENCY_VERSIONS)
    cfg = {k: v for k, v in CONFIG.items() if k not in ("OUTPUT_ROOT", "CACHE_DIR")}
    cfg["DATASET_PATH"] = Path(cfg["DATASET_PATH"]).name if cfg.get("DATASET_PATH") else None  # no machine-specific paths in artifacts
    write_json(OUTPUT_DIR / "experiment_config.json", {**cfg, **(extra_config or {}), "environment": ENVIRONMENT})
    pd.DataFrame(EXCLUSION_LEDGER).to_csv(OUTPUT_DIR / "exclusion_ledger.csv", index=False)
    for name, t in (tables or {}).items():
        t.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)


EXCLUSION_LEDGER = []


def ledger(step, reason, before, after):
    """Record every row exclusion -- rows are never dropped silently."""
    EXCLUSION_LEDGER.append({"step": step, "reason": reason, "rows_before": int(before), "rows_removed": int(before - after), "rows_after": int(after)})
    print(f"[exclusion] {step}: {reason}: removed {before - after:,} of {before:,} -> {after:,} remain")


def bootstrap_ci(y_true, y_pred, metric, n_boot=500, seed=0, alpha=0.05, stratify=None):
    """Percentile bootstrap CI for metric(y_true, y_pred), resampling test rows (optionally stratified by a label vector)."""
    rng = np.random.default_rng(seed)
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    n = len(y_true)
    idx_by = [np.where(np.asarray(stratify) == k)[0] for k in np.unique(stratify)] if stratify is not None else None
    vals = []
    for _ in range(n_boot):
        idx = np.concatenate([rng.choice(ix, size=len(ix), replace=True) for ix in idx_by]) if idx_by else rng.integers(0, n, n)
        try:
            vals.append(metric(y_true[idx], y_pred[idx]))
        except ValueError:
            continue
    lo, hi = np.percentile(vals, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(lo), float(hi)

In [ ]:
# ======================= description-derived features (no fitting; safe to compute before splitting) =======================
FLAG_PATTERNS = {
    "kw_rce": r"remote code execution|execute arbitrary (?:code|commands?)|arbitrary code|code execution|command injection|os command",
    "kw_sqli": r"sql injection|\bsqli\b",
    "kw_xss": r"cross[- ]site scripting|\bxss\b",
    "kw_memory": r"buffer overflow|out[- ]of[- ]bounds|heap[- ]based|stack[- ]based|use[- ]after[- ]free|memory corruption",
    "kw_auth_bypass": r"authentication bypass|bypass authentication|improper authentication|missing authentication",
    "kw_privesc": r"privilege escalation|escalat\w+ privileges|gain privileges|elevat\w+ privileges",
    "kw_dos": r"denial of service|\bdos\b|infinite loop|resource exhaustion",
    "kw_traversal": r"path traversal|directory traversal",
    "kw_csrf": r"cross[- ]site request forgery|\bcsrf\b",
    "kw_infoleak": r"information disclosure|sensitive information|information leak",
    "kw_remote": r"\bremote(?:ly)?\b|over the network",
    "kw_local": r"\blocal(?:ly)?\b",
    "kw_unauth": r"unauthenticated|without authentication|not require authentication|no authentication",
    "kw_auth": r"\bauthenticated\b|valid credentials|logged[- ]in|low[- ]privileged",
    "kw_user_interaction": r"user interaction|convince|trick|if a user|victim|\bclick",
    "kw_exploit_public": r"exploit (?:is|has been|code)[^.]{0,30}(?:public|disclosed|available|released)|publicly (?:exploit|disclosed)|proof[- ]of[- ]concept",
}
FLAG_COLUMNS = list(FLAG_PATTERNS)


def add_text_features(df):
    """Adds desc_length, desc_words and kw_* flags from description_en. Pure row-wise functions of the text -> no leakage."""
    d = df.copy()
    text = d["description_en"].fillna("")
    low = text.str.lower()
    d["desc_length"] = text.str.len().astype("int32")
    d["desc_words"] = text.str.split().str.len().astype("int32")
    for name, pat in FLAG_PATTERNS.items():
        d[name] = low.str.contains(pat, regex=True).astype("int8")
    return d


def normalize_description(s):
    """Normalisation used for near-duplicate detection: lower-case, digits -> '#', collapse whitespace."""
    return s.fillna("").str.lower().str.replace(r"\d+", "#", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()

## 5. Dataset discovery and loading
Discovery order: explicit `DATASET_PATH` → `/kaggle/input` → cache dir → working dir. A local source always wins; the network is touched only if nothing is found **and** `ALLOW_NETWORK=True`.

In [ ]:
df_raw, DATA_META = load_dataset(need_exploit_signals=False)
if CONFIG["FAST_DEV"]:
    df_raw = df_raw.sample(CONFIG["FAST_DEV_ROWS"], random_state=CONFIG["SEED"]).reset_index(drop=True)
    print(f"!! FAST_DEV: using a random {len(df_raw):,}-row subsample. Do NOT report results from this run.")
DATA_META["fast_dev"] = CONFIG["FAST_DEV"]

print("\n================ SETUP SUMMARY ================")
print(f"environment       : {ENVIRONMENT}")
print(f"python / libs     : {DEPENDENCY_VERSIONS['python']} | numpy {DEPENDENCY_VERSIONS['numpy']} | pandas {DEPENDENCY_VERSIONS['pandas']} | sklearn {DEPENDENCY_VERSIONS['sklearn']}")
print(f"optional packages : {OPTIONAL_AVAILABLE}")
print(f"data source       : {DATA_META['source_kind']} -- {DATA_META['source_description']}")
for f in DATA_META["files"]:
    print(f"  file            : {f['name']}  {f['bytes']/1e6:.1f} MB  sha256={f['sha256'][:16]}...")
print(f"dataset fingerprint: {DATA_META['dataset_fingerprint_sha256']}")
print(f"rows x cols       : {df_raw.shape}")
print(f"published range   : {df_raw['published_date'].min()} -> {df_raw['published_date'].max()}")
print(f"output dir        : {OUTPUT_DIR}")
print(f"cache dir         : {CONFIG['CACHE_DIR']}")
print(f"target            : {TV_NAME} base score / qualitative band")
print("================================================")

## 6. Schema and data-quality validation
Every exclusion below is recorded in `EXCLUSION_LEDGER` (written to `exclusion_ledger.csv`). **No row is dropped silently.**

In [ ]:
# --- 6.1 schema check -------------------------------------------------------------
CORE_REQUIRED = ["cve_id", "published_date", "description_en", "vuln_status", "source_identifier", "primary_cwe_id",
                 "reference_count", "cpe_match_count"]
CVSS_REQUIRED = [f"cvss_{v}_{f}" for v in CVSS_VERSIONS for f in ("vector", "base_score", "base_severity", "type")]
missing_cols = [c for c in CORE_REQUIRED + CVSS_REQUIRED if c not in df_raw.columns]
if missing_cols:
    raise ValueError(
        f"Schema mismatch: {len(missing_cols)} required column(s) missing: {missing_cols}.\n"
        "This notebook expects the canonical schema of Kaggle 'afr1ste/nvd-cve-cvss-kev-vulnerabilities-2020-2026' "
        "(nvd_cve_records.parquet) or official NVD CVE 2.0 feeds. Rename columns or set CONFIG['DATASET_PATH'] accordingly."
    )
for v in CVSS_VERSIONS:  # numeric coercion, loudly
    col = f"cvss_{v}_base_score"
    n_before = df_raw[col].notna().sum()
    df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")
    if df_raw[col].notna().sum() != n_before:
        print(f"WARNING: {n_before - df_raw[col].notna().sum()} non-numeric values in {col} coerced to NaN")
print("schema OK:", len(CORE_REQUIRED + CVSS_REQUIRED), "required columns present;", df_raw.shape[1], "columns total")

# --- 6.2 missingness & duplicates -------------------------------------------------
miss = (df_raw[CORE_REQUIRED + CVSS_REQUIRED].isna().mean() * 100).sort_values(ascending=False).rename("pct_missing").to_frame()
display(miss.head(15))
print(f"duplicate CVE ids: {int(df_raw['cve_id'].duplicated().sum()):,}")

# --- 6.3 cleaning with an exclusion ledger -----------------------------------------
df = df_raw.copy()
n = len(df); df = df[df["cve_id"].astype(str).str.match(CVE_ID_RE)]; ledger("cve_id", "malformed CVE identifier (not CVE-YYYY-NNNN+)", n, len(df))
n = len(df); df = df.sort_values("last_modified_date").drop_duplicates("cve_id", keep="last"); ledger("duplicates", "duplicate CVE id (kept most recently modified)", n, len(df))
n = len(df); df = df[df["vuln_status"].fillna("") != "Rejected"]; ledger("status", "NVD vuln_status == Rejected (not a vulnerability)", n, len(df))
n = len(df); df = df[df["description_en"].fillna("").str.strip().ne("")]; ledger("description", "missing/empty English description", n, len(df))
n = len(df); df = df[df["published_date"].notna()]; ledger("published", "missing/unparsable published date", n, len(df))
df = df.reset_index(drop=True)
print(f"\nclean CVE table: {len(df):,} rows")

In [ ]:
# --- 6.4 per-version CVSS consistency audit (versions are audited separately; never pooled) -------------
BASE_METRICS = {  # base metrics only; temporal/environmental/supplemental metrics are excluded on purpose
    "v31": ["AV", "AC", "PR", "UI", "S", "C", "I", "A"], "v30": ["AV", "AC", "PR", "UI", "S", "C", "I", "A"],
    "v2": ["AV", "AC", "Au", "C", "I", "A"], "v40": ["AV", "AC", "AT", "PR", "UI", "VC", "VI", "VA", "SC", "SI", "SA"],
}


def invalid_reasons(d, v):
    """Boolean frame, one column per defect, for rows that carry any CVSS-v data. A row may have several defects."""
    vec, score, lab = d[f"cvss_{v}_vector"], d[f"cvss_{v}_base_score"], d[f"cvss_{v}_base_severity"]
    parsed = vec.map(parse_cvss_vector)
    r = pd.DataFrame(index=d.index)
    r["vector_missing_or_wrong_version"] = parsed.map(lambda p: p is None or p[0] != CVSS_VERSIONS[v])
    r["score_missing_or_out_of_range"] = ~score.between(0, 10)
    r["metrics_incomplete"] = parsed.map(lambda p: p is None or not all(k in p[1] for k in BASE_METRICS[v]))
    r["severity_label_mismatch"] = pd.Series([cvss_severity(s, v) for s in score], index=d.index).fillna("?") != lab.astype(str).str.upper()
    if v in OFFICIAL_CALCULATORS:
        r["official_calculator_mismatch"] = vec.map(OFFICIAL_CALCULATORS[v]) != score
    return r


audit_rows = []
for v in CVSS_VERSIONS:
    has = df[f"cvss_{v}_vector"].notna() | df[f"cvss_{v}_base_score"].notna()
    d = df[has]
    rs = invalid_reasons(d, v)
    audit_rows.append({"version": f"CVSS v{CVSS_VERSIONS[v]}", "rows_with_data": len(d), **{k: int(rs[k].sum()) for k in rs.columns},
                       "rows_with_any_defect": int(rs.any(axis=1).sum()),
                       "official_calculator": "implemented" if v in OFFICIAL_CALCULATORS else "NOT implemented (v4.0 macro-vector table)"})
version_audit = pd.DataFrame(audit_rows).fillna("n/a")
display(version_audit)

# --- 6.5 restrict to the target version; remove invalid rows with a documented reason -----------------
n = len(df); d_tv = df[df[f"cvss_{TV}_base_score"].notna() | df[f"cvss_{TV}_vector"].notna()].copy()
ledger("task scope", f"no {TV_NAME} score/vector (different CVSS version or unscored; versions are not mixed)", n, len(d_tv))
rs = invalid_reasons(d_tv, TV)
for reason in rs.columns:
    print(f"  defect '{reason}': {int(rs[reason].sum()):,} rows")
n = len(d_tv); d_tv = d_tv[~rs.any(axis=1)]; ledger("invalid score", f"{TV_NAME}: any of {list(rs.columns)}", n, len(d_tv))
d_tv = d_tv.sort_values("published_date").reset_index(drop=True)
d_tv["label"] = d_tv[f"cvss_{TV}_base_score"].map(lambda s: cvss_severity(s, TV))
d_tv["y"] = d_tv[f"cvss_{TV}_base_score"].astype(float)
d_tv["offensivex_severity"] = d_tv["label"].map(CVSS_TO_OFFENSIVEX_SEVERITY)
print(f"\n{TV_NAME} analysis table: {len(d_tv):,} rows, published {d_tv['published_date'].min().date()} -> {d_tv['published_date'].max().date()}")
display(pd.DataFrame(EXCLUSION_LEDGER))
assert len(d_tv) > 1000, "Too few rows after validation to train anything meaningful -- check the dataset/TARGET_VERSION."

### 6.6 Validator self-test (proves the checks above can actually fail)
The real data is clean (0 defects), which would make the checks above look vacuous. Here we inject a **handful of deliberately corrupted rows into a tiny throw-away frame** and confirm each defect type is detected. This frame is never used for training or evaluation.

In [ ]:
_probe = d_tv.head(6).copy()
_probe.loc[1, f"cvss_{TV}_base_score"] = 11.5                              # out of range
_probe.loc[2, f"cvss_{TV}_base_severity"] = "LOW" if _probe.loc[2, "label"] != "LOW" else "HIGH"   # label inconsistent with score
_probe.loc[3, f"cvss_{TV}_vector"] = "CVSS:2.0/AV:N"                       # wrong version prefix / incomplete
_probe.loc[4, f"cvss_{TV}_base_score"] = float(_probe.loc[4, f"cvss_{TV}_base_score"]) - 0.1     # score != official formula(vector)
_probe.loc[5, f"cvss_{TV}_vector"] = None                                   # missing vector
_st = invalid_reasons(_probe, TV)
display(_st.astype(int))
assert not _st.iloc[0].any(), "clean row must pass"
assert _st.iloc[1]["score_missing_or_out_of_range"] and _st.iloc[2]["severity_label_mismatch"] and _st.iloc[3]["vector_missing_or_wrong_version"]
assert _st.iloc[4]["official_calculator_mismatch"] and _st.iloc[5]["vector_missing_or_wrong_version"]
print("validator self-test passed: all injected defect types detected, clean row accepted.")
del _probe, _st

## 7. Exploratory data analysis
Distributions use all rows; scatter-style plots use a fixed random sample (`PLOT_SAMPLE_ROWS`) for speed. Plots are displayed and saved to `plots/`.

In [ ]:
d_tv = add_text_features(d_tv)
d_tv["pub_year"] = d_tv["published_date"].dt.year
CLASS_ORDER = ["NONE", "LOW", "MEDIUM", "HIGH", "CRITICAL"] if TV != "v2" else ["LOW", "MEDIUM", "HIGH"]
class_counts = d_tv["label"].value_counts().reindex(CLASS_ORDER).fillna(0).astype(int)
print(class_counts.to_string())

# 1) target-score distribution -----------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(d_tv["y"], bins=np.arange(0, 10.25, 0.1), color="#3b6ea5")
for t, name in [(4.0, "MEDIUM >= 4.0"), (7.0, "HIGH >= 7.0"), (9.0, "CRITICAL >= 9.0")]:
    ax.axvline(t, color="crimson", ls="--", lw=1)
    ax.text(t + 0.05, ax.get_ylim()[1] * 0.92, name, rotation=90, va="top", fontsize=8, color="crimson")
ax.set(title=f"{TV_NAME} base-score distribution (all {len(d_tv):,} analysed CVEs)", xlabel="CVSS base score (0-10)", ylabel="CVE count")
savefig("01_target_score_distribution")

# 2) severity-class distribution over time ---------------------------------------------
ct = pd.crosstab(d_tv["pub_year"], d_tv["label"]).reindex(columns=[c for c in CLASS_ORDER if c in set(d_tv["label"])])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ct.plot(kind="bar", stacked=True, ax=axes[0], colormap="viridis")
axes[0].set(title="Severity class counts by publication year", xlabel="Publication year", ylabel="CVE count")
(ct.div(ct.sum(axis=1), axis=0) * 100).plot(kind="bar", stacked=True, ax=axes[1], colormap="viridis", legend=False)
axes[1].set(title="Severity class share by publication year", xlabel="Publication year", ylabel="% of CVEs published that year")
savefig("02_severity_class_distribution_by_year")

# 3) class imbalance ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 4))
cc = class_counts[class_counts > 0]
ax.bar(cc.index, cc.values, color="#8a5a9e")
ax.set_yscale("log")
for i, (k, v) in enumerate(cc.items()):
    ax.text(i, v, f"{v:,}\n({v / cc.sum():.1%})", ha="center", va="bottom", fontsize=8)
ax.set(title=f"Class imbalance (majority:minority = {cc.max() / cc.min():,.0f}:1, excluding empty classes)", xlabel="Severity class", ylabel="CVE count (log scale)")
savefig("03_class_imbalance")

# 4) missingness ------------------------------------------------------------------------
key_cols = ["description_en", "primary_cwe_id", "source_identifier", "reference_count", "cpe_match_count",
            "cvss_v40_base_score", "cvss_v31_base_score", "cvss_v30_base_score", "cvss_v2_base_score"]
cov = df.assign(pub_year=df["published_date"].dt.year).groupby("pub_year")[[f"cvss_{v}_base_score" for v in CVSS_VERSIONS]].agg(lambda s: s.notna().mean() * 100)
cov.columns = [f"CVSS v{CVSS_VERSIONS[c.split('_')[1]]}" for c in cov.columns]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
(df[key_cols].isna().mean() * 100).sort_values().plot(kind="barh", ax=axes[0], color="#c0504d")
axes[0].set(title="Missing values (% of clean CVE table)", xlabel="% missing")
sns.heatmap(cov.T, annot=True, fmt=".0f", cmap="Blues", ax=axes[1], cbar_kws={"label": "% of CVEs with a score"})
axes[1].set(title="CVSS-version coverage by publication year (why versions cannot be pooled)", xlabel="Publication year", ylabel="")
savefig("04_missingness_and_version_coverage")

# 5) feature distributions --------------------------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes[0, 0].hist(d_tv["desc_length"].clip(upper=d_tv["desc_length"].quantile(0.99)), bins=60, color="#3b6ea5"); axes[0, 0].set(title="Description length (chars, clipped at p99)", xlabel="characters", ylabel="CVE count")
axes[0, 1].hist(np.log1p(d_tv["reference_count"]), bins=40, color="#5b9a5b"); axes[0, 1].set(title="Reference count (log1p) -- post-publication enrichment", xlabel="log(1 + references)", ylabel="CVE count")
axes[0, 2].hist(np.log1p(d_tv["cpe_match_count"]), bins=40, color="#5b9a5b"); axes[0, 2].set(title="CPE-match count (log1p) -- post-publication enrichment", xlabel="log(1 + CPE matches)", ylabel="CVE count")
d_tv["primary_cwe_id"].fillna("(missing)").value_counts().head(12).iloc[::-1].plot(kind="barh", ax=axes[1, 0], color="#d08a3b"); axes[1, 0].set(title="Top-12 primary CWE", xlabel="CVE count")
d_tv["source_identifier"].fillna("(missing)").value_counts().head(10).iloc[::-1].plot(kind="barh", ax=axes[1, 1], color="#d08a3b"); axes[1, 1].set(title="Top-10 reporting CNA / source", xlabel="CVE count")
(d_tv[FLAG_COLUMNS].mean() * 100).sort_values().plot(kind="barh", ax=axes[1, 2], color="#7a7a7a"); axes[1, 2].set(title="Description keyword-flag prevalence", xlabel="% of CVEs")
savefig("05_feature_distributions")

# 6) correlation heatmap of eligible numeric features (+ target for reference) -------------------
num_eda = ["desc_length", "desc_words"] + FLAG_COLUMNS + ["reference_count", "cpe_match_count", "pub_year", "y"]
corr = d_tv[num_eda].sample(min(len(d_tv), 100_000), random_state=CONFIG["SEED"]).corr(method="spearman")
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, cmap="coolwarm", center=0, vmin=-1, vmax=1, ax=ax, cbar_kws={"label": "Spearman rho"}, square=True, linewidths=0.2)
ax.set_title(f"Spearman correlation of numeric features with each other and the target 'y' ({TV_NAME} base score)\n(EDA only: the target row/column is shown for reference and is never a model input)")
savefig("06_correlation_heatmap")
display(corr["y"].drop("y").sort_values(key=np.abs, ascending=False).head(8).to_frame("spearman_with_target"))

## 8. Target and feature definitions

**Targets** (documented scoring definition): `y` = the *reported* CVSS base score of the chosen version (regression, 0.0–10.0); `label` = the qualitative band of that version derived from the score by the specification's thresholds (classification). Both come from the same record, so they are consistent by construction (verified in §6.5).

**Experiment A — reconstruction benchmark.** Inputs: the version's *base metric values parsed from the vector string* (categorical). Eligible because they are the official definition inputs. Excluded even here: the reported `exploitability_score` / `impact_score` sub-scores (direct score components) and every other column.
**At prediction time you would need:** the complete CVSS base vector — i.e. the analyst has already done the scoring. Hence "not evidence of discovering severity".

**Experiment B — metadata-based prediction.** Information set = *what exists when a CNA first publishes a CVE record*:
* `description_en` (text), `source_identifier` (reporting CNA), `primary_cwe_id` (CWE assigned by the CNA/NVD — may be added slightly after publication; see ablation), text-derived flags/length.
* **Explicitly excluded:** every `cvss_*` column of **any** version (target or target-adjacent; e.g. v2 score would trivialise a v3.1 prediction), `best_cvss_*`, exploitability/impact sub-scores, `kev_*`, `epss_*` (outcomes/estimates formed after publication), `vuln_status` and `last_modified_date` (set by post-publication analysis), `weakness_count` (counted differently in different sources).
* **Ablation only:** `reference_count`, `cpe_match_count` — these accumulate *after* publication during NVD enrichment (the same analysis that produces the NVD CVSS score), so they are a mild leak. Feature set `core_plus_enrichment` quantifies the effect (RQ3).

**When would Experiment B's model be used?** At triage time for a *freshly published, not-yet-scored CVE* (e.g. to queue it before NVD/CNA scoring). It would never replace the official score once available.

In [ ]:
TEXT_COL = "description_en"
CAT_COLS = ["source_identifier", "primary_cwe_id"]
NUM_CORE = ["desc_length", "desc_words"] + FLAG_COLUMNS
NUM_ENRICH = ["reference_count", "cpe_match_count"]
FEATURE_SETS = {
    "text_only": dict(text=True, cat=[], num=[]),
    "core": dict(text=True, cat=CAT_COLS, num=NUM_CORE),
    "core_plus_enrichment": dict(text=True, cat=CAT_COLS, num=NUM_CORE + NUM_ENRICH),
}
HEADLINE_SET = "core"
FORBIDDEN_RE = re.compile(r"cvss|score|severity|epss|kev|^y$|^label$|best_|exploitab|impact|vuln_status|last_modified|offensivex_severity", re.I)


def feature_columns(spec):
    return ([TEXT_COL] if spec["text"] else []) + spec["cat"] + spec["num"]


for name, spec in FEATURE_SETS.items():
    bad = [c for c in feature_columns(spec) if FORBIDDEN_RE.search(c)]
    assert not bad, f"LEAKAGE: feature set {name} contains forbidden columns {bad}"
    print(f"feature set {name:22s}: {len(feature_columns(spec)):3d} input columns  (leakage name-audit passed)")

## 9. Leakage analysis
The table records, for every dataset column group, whether it may be used and why. The name-based audit above is a tripwire; the reasoning below is the real control.

In [ ]:
leakage_table = pd.DataFrame([
    ("cvss_<any version>_vector / base_score / base_severity / sub-scores", "target or target-derived", "Exp A: metrics from the vector (benchmark only). Exp B: EXCLUDED", "CVSS score = f(vector); other versions' scores are target-adjacent"),
    ("best_cvss_* (preferred-version fields)", "target-derived, mixes versions", "EXCLUDED", "Mixes CVSS versions with different semantics"),
    ("cvss_*_type / cvss_*_source (scorer identity)", "post-hoc metadata", "subgroup analysis only", "Known only once scoring is done"),
    ("kev_*, epss_*", "post-publication outcome/estimate", "EXCLUDED", "Exploitation observed/estimated later; unrelated to severity definition"),
    ("vuln_status, last_modified_date", "post-publication processing state", "EXCLUDED", "Analyzed/Deferred status correlates with who scored it"),
    ("reference_count, cpe_match_count", "post-publication enrichment (mild leak)", "ablation `core_plus_enrichment` only", "Accumulate as NVD analysts enrich the record"),
    ("weakness_count", "inconsistent definition across sources", "EXCLUDED", "73% agreement between Kaggle build and official-feed parse"),
    ("description_en (+ derived length/flags)", "available at publication (CNA-authored)", "INCLUDED", "Descriptions may be edited later; accepted limitation"),
    ("source_identifier (CNA)", "available at publication", "INCLUDED", "CNAs have systematic scoring habits -- real signal, not leakage"),
    ("primary_cwe_id", "usually available at/near publication", "INCLUDED (see caveat)", "NVD may add/replace CWE after publication"),
    ("published_date", "split key", "NOT a feature", "Year as a feature is non-stationary under a chronological split"),
], columns=["column group", "temporal status", "decision", "reason"])
display(leakage_table)

# --- duplicate-CVE and near-duplicate (templated-description) checks ------------------------------
d_tv["desc_norm"] = normalize_description(d_tv["description_en"])
grp_size = d_tv["desc_norm"].map(d_tv["desc_norm"].value_counts())
dup_mask = grp_size > 1
lab_agree = d_tv[dup_mask].groupby("desc_norm")["label"].agg(lambda s: s.nunique() == 1).mean() if dup_mask.any() else float("nan")
print(f"duplicate CVE ids in analysis table          : {int(d_tv['cve_id'].duplicated().sum())}")
print(f"rows whose normalised description is shared  : {dup_mask.sum():,} ({dup_mask.mean():.1%}) in {d_tv.loc[dup_mask, 'desc_norm'].nunique():,} groups")
print(f"…of those groups, share with one single label: {lab_agree:.1%}  (high => templated descriptions => easy-to-memorise rows)")

## 10. Train / validation / test strategy
* **Primary (headline) protocol — chronological:** CVEs sorted by `published_date`; first 70 % train, next 15 % validation (model selection / tuning), last 15 % test. Measures generalisation to *later* CVEs, which is the real deployment condition.
* **Secondary — random stratified split** (same sizes): shown for comparison and for repeated-split variability. It is typically optimistic because neighbours in time share scorers, templates and vocabulary.
* All preprocessing (TF-IDF, SVD, encoders, scalers) is fit on the training portion **inside scikit-learn pipelines**. The test split is touched once per model for reporting; model/hyper-parameter selection uses validation only.
* Classes with fewer than `MIN_CLASS_ROWS` rows are excluded from *classification* (kept for regression) with the reason recorded.

In [ ]:
from sklearn.model_selection import train_test_split


def chrono_split(n, fr):
    i1, i2 = int(n * fr[0]), int(n * (fr[0] + fr[1]))
    return np.arange(0, i1), np.arange(i1, i2), np.arange(i2, n)


assert d_tv["published_date"].is_monotonic_increasing
tr_idx, va_idx, te_idx = chrono_split(len(d_tv), CONFIG["SPLIT_FRACTIONS"])
vc = d_tv["label"].value_counts()
CLASSES = [c for c in CLASS_ORDER if vc.get(c, 0) >= CONFIG["MIN_CLASS_ROWS"]]
dropped_classes = {c: int(vc.get(c, 0)) for c in CLASS_ORDER if 0 < vc.get(c, 0) < CONFIG["MIN_CLASS_ROWS"]}
if dropped_classes:
    n_cls_drop = int(d_tv["label"].isin(dropped_classes).sum())
    EXCLUSION_LEDGER.append({"step": "classification only", "reason": f"classes with < {CONFIG['MIN_CLASS_ROWS']} rows cannot be learned/evaluated: {dropped_classes}",
                             "rows_before": len(d_tv), "rows_removed": n_cls_drop, "rows_after": len(d_tv) - n_cls_drop})
    print(f"classification excludes {n_cls_drop} rows of classes {dropped_classes} (regression keeps them)")
CLS_ID = {c: i for i, c in enumerate(CLASSES)}
d_tv["y_cls"] = d_tv["label"].map(CLS_ID)  # NaN for excluded classes

rows = []
for nm, ix in [("train", tr_idx), ("validation", va_idx), ("test", te_idx)]:
    sub = d_tv.iloc[ix]
    rows.append({"split": nm, "rows": len(sub), "from": sub["published_date"].min().date(), "to": sub["published_date"].max().date(),
                 **{f"{c} %": round(100 * (sub["label"] == c).mean(), 2) for c in CLASSES}, "mean score": round(sub["y"].mean(), 3)})
split_table = pd.DataFrame(rows)
display(split_table)

# near-duplicate overlap between train and test (text seen in training) -> used as a sensitivity subgroup
train_norm = set(d_tv.iloc[tr_idx]["desc_norm"])
d_tv["text_seen_in_train"] = d_tv["desc_norm"].isin(train_norm)
print(f"test rows whose normalised description also appears in train: {d_tv.iloc[te_idx]['text_seen_in_train'].mean():.1%}")


def random_split(seed):
    """Stratified random 70/15/15 over ALL rows (classes too small for stratification are pooled as 'RARE')."""
    strat = d_tv["label"].where(d_tv["label"].isin(CLASSES), "RARE")
    idx = np.arange(len(d_tv))
    a, te = train_test_split(idx, test_size=CONFIG["SPLIT_FRACTIONS"][2], stratify=strat, random_state=seed)
    tr, va = train_test_split(a, test_size=CONFIG["SPLIT_FRACTIONS"][1] / (1 - CONFIG["SPLIT_FRACTIONS"][2]), stratify=strat.iloc[a], random_state=seed)
    return np.sort(tr), np.sort(va), np.sort(te)

## 11. Baselines and metric helpers
Baseline for every task: a **Dummy** model (predict the training mean / majority class). Any candidate must beat it on the *validation* split before its test number means anything.

In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score, classification_report, confusion_matrix,
                             f1_score, mean_absolute_error, mean_squared_error, precision_recall_curve, precision_recall_fscore_support,
                             r2_score, roc_auc_score, roc_curve)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, label_binarize

if tuple(int(x) for x in sklearn.__version__.split(".")[:2]) < (1, 3):
    raise ImportError(f"scikit-learn >= 1.3 required (found {sklearn.__version__}).")
N_JOBS = CONFIG["N_JOBS"]
SEED = CONFIG["SEED"]


def score_to_band(scores, v):
    """Vectorised version of cvss_severity() applied to (rounded) predicted scores."""
    s = np.round(np.clip(np.asarray(scores, float), 0, 10), 1)
    if v == "v2":
        return np.select([s < 4.0, s < 7.0], ["LOW", "MEDIUM"], "HIGH")
    return np.select([s == 0.0, s < 4.0, s < 7.0, s < 9.0], ["NONE", "LOW", "MEDIUM", "HIGH"], "CRITICAL")


def reg_metrics(y, p):
    y, p = np.asarray(y, float), np.clip(np.asarray(p, float), 0, 10)  # scores live in [0, 10]; clipping is part of the model contract
    r = y - p
    return {"MAE": float(np.mean(np.abs(r))), "RMSE": float(np.sqrt(mean_squared_error(y, p))), "R2": float(r2_score(y, p)),
            "resid_mean": float(r.mean()), "resid_std": float(r.std()), "resid_p05": float(np.percentile(r, 5)), "resid_p95": float(np.percentile(r, 95)),
            "within_0.05": float(np.mean(np.abs(r) < 0.05)), "within_0.5": float(np.mean(np.abs(r) <= 0.5))}


def cls_metrics(y, p, labels):
    pm, rm, fm, _ = precision_recall_fscore_support(y, p, labels=labels, average="macro", zero_division=0)
    pw, rw, fw, _ = precision_recall_fscore_support(y, p, labels=labels, average="weighted", zero_division=0)
    return {"accuracy": float(accuracy_score(y, p)), "balanced_accuracy": float(balanced_accuracy_score(y, p)),
            "precision_macro": float(pm), "recall_macro": float(rm), "f1_macro": float(fm),
            "precision_weighted": float(pw), "recall_weighted": float(rw), "f1_weighted": float(fw)}


mae = lambda y, p: float(np.mean(np.abs(np.asarray(y, float) - np.clip(np.asarray(p, float), 0, 10))))

## 12. Experiment A — metric-based **reconstruction benchmark** (NOT a predictive-capability claim)
**What this measures:** how well a generic ML model can re-learn the deterministic CVSS formula from the eight/six/eleven base metrics. **Why it matters here:** it is the *ceiling* that any metadata-based model (Experiment B) is trying to approach without being handed the metrics, and it shows that an exact calculator beats every ML model at this task — so in OffensiveX, scores that have a CVSS vector must keep using the deterministic calculation.
Each CVSS version is evaluated **separately** with its own metric set. v4.0 has no official calculator implemented in this notebook (its score comes from a macro-vector lookup), so for v4.0 only the ML-vs-reported comparison is shown.

In [ ]:
def metric_frame(d, v):
    parsed = d[f"cvss_{v}_vector"].map(lambda s: parse_cvss_vector(s)[1])
    return pd.DataFrame(list(parsed), index=d.index)[BASE_METRICS[v]].astype(str)


def recon_model(kind, cols):
    pre = ColumnTransformer([("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cols)])
    est = {"dummy_mean": DummyRegressor(strategy="mean"),
           "ridge": Ridge(alpha=1.0),
           "random_forest": RandomForestRegressor(n_estimators=100, min_samples_leaf=2, n_jobs=N_JOBS, random_state=SEED),
           "hist_gradient_boosting": HistGradientBoostingRegressor(max_iter=300, learning_rate=0.1, random_state=SEED)}[kind]
    return Pipeline([("prep", pre), ("model", est)])


RECON_MODELS = ["dummy_mean", "ridge", "random_forest", "hist_gradient_boosting"]
RECON_ROWS, RECON_PREDS = [], {}
for v in CVSS_VERSIONS:
    has = df[f"cvss_{v}_vector"].notna() | df[f"cvss_{v}_base_score"].notna()
    dv = df[has]
    bad = invalid_reasons(dv, v).any(axis=1)
    dv = dv[~bad].sort_values("published_date")
    if len(dv) < CONFIG["MIN_ROWS_PER_VERSION"]:
        print(f"CVSS v{CVSS_VERSIONS[v]}: only {len(dv)} valid rows (< {CONFIG['MIN_ROWS_PER_VERSION']}) -> skipped, reason recorded")
        RECON_ROWS.append({"version": f"CVSS v{CVSS_VERSIONS[v]}", "model": "(skipped: too few rows)", "n_test": 0})
        continue
    X, y = metric_frame(dv, v), dv[f"cvss_{v}_base_score"].astype(float).values
    itr, iva, ite = chrono_split(len(dv), CONFIG["SPLIT_FRACTIONS"])
    sev = dv[f"cvss_{v}_base_severity"].astype(str).str.upper().values
    for kind in RECON_MODELS:
        m = recon_model(kind, list(X.columns)).fit(X.iloc[itr], y[itr])
        p = m.predict(X.iloc[ite])
        met = reg_metrics(y[ite], p)
        band = score_to_band(p, v)
        met.update({"band_accuracy": float(np.mean(band == sev[ite])), "band_f1_macro": float(f1_score(sev[ite], band, average="macro", zero_division=0))})
        RECON_ROWS.append({"version": f"CVSS v{CVSS_VERSIONS[v]}", "model": kind, "n_train": len(itr), "n_test": len(ite), **met})
        if v == TV:
            RECON_PREDS[kind] = p
    if v in OFFICIAL_CALCULATORS:  # the official formula applied to the same test rows
        c = dv[f"cvss_{v}_vector"].iloc[ite].map(OFFICIAL_CALCULATORS[v]).values.astype(float)
        met = reg_metrics(y[ite], c)
        band = score_to_band(c, v)
        met.update({"band_accuracy": float(np.mean(band == sev[ite])), "band_f1_macro": float(f1_score(sev[ite], band, average="macro", zero_division=0))})
        RECON_ROWS.append({"version": f"CVSS v{CVSS_VERSIONS[v]}", "model": "OFFICIAL CALCULATOR", "n_train": 0, "n_test": len(ite), **met})
        if v == TV:
            RECON_PREDS["OFFICIAL CALCULATOR"] = c
    if v == TV:
        RECON_TEST = dict(y=y[ite], sev=sev[ite], X=X.iloc[ite], n_train=len(itr))
    print(f"CVSS v{CVSS_VERSIONS[v]}: {len(dv):,} valid rows ({int(bad.sum())} invalid removed), train/val/test = {len(itr):,}/{len(iva):,}/{len(ite):,}")
recon_df = pd.DataFrame(RECON_ROWS)
show = [c for c in ["version", "model", "n_test", "MAE", "RMSE", "R2", "within_0.05", "within_0.5", "band_accuracy", "band_f1_macro"] if c in recon_df.columns]
display(recon_df[show].round(4))

In [ ]:
# --- A) comparison with the official calculator on the target version (chronological test split) -------------------
best_recon = recon_df[(recon_df.version == TV_NAME) & (~recon_df.model.isin(["OFFICIAL CALCULATOR", "dummy_mean"]))].sort_values("MAE").iloc[0]["model"]
p_best = np.clip(RECON_PREDS[best_recon], 0, 10)
yt = RECON_TEST["y"]
cis = {k: bootstrap_ci(yt, np.clip(RECON_PREDS[k], 0, 10), mae, n_boot=CONFIG["N_BOOTSTRAP"], seed=SEED) for k in RECON_PREDS}
print(f"best ML reconstruction model on {TV_NAME}: {best_recon}")
if "OFFICIAL CALCULATOR" in RECON_PREDS:
    diff = np.abs(p_best - RECON_PREDS["OFFICIAL CALCULATOR"])
    print(f"ML vs official calculator on test rows: mean |diff| = {diff.mean():.4f}, share within 0.05 = {(diff < 0.05).mean():.2%}, max = {diff.max():.2f}")
    print("-> The deterministic calculator is exact by construction; any ML residual here is pure approximation error, not insight.")

# 1) actual vs predicted ---------------------------------------------------------------------------------
rs = np.random.default_rng(SEED).choice(len(yt), size=min(len(yt), CONFIG["PLOT_SAMPLE_ROWS"]), replace=False)
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(yt[rs] + np.random.default_rng(1).normal(0, 0.03, len(rs)), p_best[rs], s=4, alpha=0.25, color="#3b6ea5")
ax.plot([0, 10], [0, 10], "r--", lw=1, label="perfect reconstruction")
ax.set(xlim=(0, 10.3), ylim=(0, 10.3), xlabel=f"Official {TV_NAME} base score (reported)", ylabel=f"{best_recon} prediction",
       title=f"Exp A (reconstruction): actual vs predicted, test split\n{len(rs):,} sampled points; x jittered +-0.03 to show discrete scores")
ax.legend()
savefig("A1_reconstruction_actual_vs_predicted")

# 2) residuals ----------------------------------------------------------------------------------------------
res = yt - p_best
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(res, bins=np.arange(-2, 2.05, 0.05), color="#8a5a9e"); axes[0].set(title=f"Exp A residuals (actual - predicted), {best_recon}\nmean={res.mean():.3f}, sd={res.std():.3f}", xlabel="residual (score points)", ylabel="CVE count")
axes[1].scatter(p_best[rs], res[rs], s=4, alpha=0.25, color="#8a5a9e"); axes[1].axhline(0, color="r", lw=1)
axes[1].set(title="Exp A residual vs predicted score", xlabel="predicted score", ylabel="residual (score points)")
savefig("A2_reconstruction_residuals")

# 3) model comparison with bootstrap CIs ---------------------------------------------------------------------
mvals = {k: float(np.mean(np.abs(yt - np.clip(RECON_PREDS[k], 0, 10)))) for k in RECON_PREDS}
order = sorted(mvals, key=mvals.get, reverse=True)
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(order, [mvals[k] for k in order], xerr=[[max(mvals[k] - cis[k][0], 0) for k in order], [max(cis[k][1] - mvals[k], 0) for k in order]], color="#3b6ea5", capsize=3)
ax.set(title=f"Exp A: test MAE by model ({TV_NAME}); bars = 95% bootstrap CI", xlabel="MAE (score points; lower is better)")
savefig("A3_reconstruction_model_comparison")

# 4) performance by CVSS version (each version evaluated on its own metrics) ----------------------------
pv = recon_df[recon_df.model.isin(["hist_gradient_boosting", "OFFICIAL CALCULATOR"]) & recon_df["MAE"].notna()]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.barplot(data=pv, x="version", y="MAE", hue="model", ax=axes[0]); axes[0].set(title="Reconstruction MAE by CVSS version (test split)", ylabel="MAE (score points)", xlabel="")
sns.barplot(data=recon_df[recon_df.model.isin(["hist_gradient_boosting", "ridge"]) & recon_df["R2"].notna()], x="version", y="R2", hue="model", ax=axes[1]); axes[1].set(title="Reconstruction R2 by CVSS version", ylabel="R2", xlabel="")
savefig("A4_reconstruction_by_cvss_version")

# 5) derived severity band vs official band --------------------------------------------------------------------
bands = [b for b in CLASS_ORDER if b in set(RECON_TEST["sev"]) | set(score_to_band(p_best, TV))]
cm = confusion_matrix(RECON_TEST["sev"], score_to_band(p_best, TV), labels=bands)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=bands, yticklabels=bands, ax=ax)
ax.set(title=f"Exp A: severity band derived from ML-reconstructed score\nvs official band ({best_recon}, test)", xlabel="derived band (from predicted score)", ylabel="official band")
savefig("A5_reconstruction_band_confusion")

# 6) error structure: where does the ML approximation fail? (by Scope metric) -------------------------------------
if "S" in RECON_TEST["X"].columns:
    err = pd.DataFrame({"abs_err": np.abs(res), "scope": RECON_TEST["X"]["S"].values})
    display(err.groupby("scope")["abs_err"].agg(["mean", "max", "count"]).round(4).rename(columns={"mean": "MAE", "max": "max abs error"}))
RECON_SUMMARY = {"best_model": best_recon, "best_MAE": float(np.mean(np.abs(res))), "best_R2": float(r2_score(yt, p_best)),
                 "calculator_exact": bool("OFFICIAL CALCULATOR" in RECON_PREDS and np.mean(np.abs(yt - RECON_PREDS["OFFICIAL CALCULATOR"])) == 0.0)}
print(RECON_SUMMARY)

## 13. Experiment B — metadata-based prediction: candidates, tuning, held-out evaluation
**Pipeline design (no train/test contamination):** a `ColumnTransformer` is *fit on the training rows only*. Two families share the same inputs:
* **linear** (Ridge / Logistic Regression): sparse TF-IDF(1–2-grams) + one-hot(CNA, CWE) + standardised numeric flags;
* **tree** (Random Forest / HistGradientBoosting): TF-IDF → TruncatedSVD(64) dense text embedding + one-hot + raw numeric flags.

Baselines: `DummyRegressor(mean)`, `DummyClassifier(prior)`. Hyper-parameters are chosen on the **validation** split (Ridge α, HGB learning-rate/leaf-count). Candidates are then compared on **test** (fit on train only, so the comparison is apples-to-apples); the *selected* model is finally refit on train+validation and re-scored on test for the exported artifact.
Class imbalance: class-balanced sample weights for HGB, `class_weight="balanced"` for Logistic Regression / Random Forest; **no resampling is used**, and nothing touches the test split before evaluation.

In [ ]:
from scipy import sparse
from sklearn.utils.class_weight import compute_sample_weight

XCOLS = sorted({c for s in FEATURE_SETS.values() for c in feature_columns(s)})
Xall = d_tv[XCOLS]
y_reg = d_tv["y"].values
y_cls_all = d_tv["y_cls"].values  # float with NaN for excluded classes


def make_prep(spec, family):
    parts = []
    if spec["text"]:
        tfidf = TfidfVectorizer(max_features=CONFIG["TFIDF_MAX_FEATURES"], ngram_range=(1, 2), min_df=5, sublinear_tf=True, dtype=np.float32)
        parts.append(("text", tfidf if family == "linear" else Pipeline([("tfidf", tfidf), ("svd", TruncatedSVD(CONFIG["SVD_COMPONENTS"], random_state=SEED))]), TEXT_COL))
    if spec["cat"]:
        parts.append(("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50, sparse_output=(family == "linear")), spec["cat"]))
    if spec["num"]:
        parts.append(("num", StandardScaler() if family == "linear" else "passthrough", spec["num"]))
    return ColumnTransformer(parts, sparse_threshold=0.3 if family == "linear" else 0.0)


def build_matrices(set_name, family, fit_idx, other_idx):
    """Fit the preprocessing on `fit_idx` rows ONLY and transform all requested index sets."""
    spec = FEATURE_SETS[set_name]
    cols = feature_columns(spec)
    prep = make_prep(spec, family).fit(Xall.iloc[fit_idx][cols])
    return prep, prep.transform(Xall.iloc[fit_idx][cols]), {k: prep.transform(Xall.iloc[ix][cols]) for k, ix in other_idx.items()}


t0 = time.time()
MATS = {}
for fam in ("linear", "tree"):
    prep, Mtr, others = build_matrices(HEADLINE_SET, fam, tr_idx, {"val": va_idx, "test": te_idx})
    MATS[fam] = dict(prep=prep, train=Mtr, val=others["val"], test=others["test"])
    print(f"[{HEADLINE_SET}/{fam}] preprocessing fit on {len(tr_idx):,} training rows -> matrix {Mtr.shape} ({'sparse' if sparse.issparse(Mtr) else 'dense'}) in {time.time()-t0:.0f}s")

# ---------------- regression ----------------
REG_GRIDS = {"ridge": [{"alpha": a} for a in (0.3, 1.0, 3.0, 10.0)],
             "hist_gradient_boosting": [{"learning_rate": lr, "max_leaf_nodes": ml} for lr in (0.1, 0.05) for ml in (31, 63)]}


def reg_estimator(kind, **kw):
    return {"dummy_mean": lambda: DummyRegressor(strategy="mean"),
            "ridge": lambda: Ridge(alpha=kw.get("alpha", 1.0), random_state=SEED),
            "random_forest": lambda: RandomForestRegressor(n_estimators=150, min_samples_leaf=3, max_features=0.33, max_samples=0.5, n_jobs=N_JOBS, random_state=SEED),
            "hist_gradient_boosting": lambda: HistGradientBoostingRegressor(max_iter=200, learning_rate=kw.get("learning_rate", 0.1), max_leaf_nodes=kw.get("max_leaf_nodes", 31),
                                                                            l2_regularization=1.0, early_stopping=False, random_state=SEED)}[kind]()


FAMILY = {"dummy_mean": "linear", "ridge": "linear", "random_forest": "tree", "hist_gradient_boosting": "tree"}
y_tr, y_va, y_te = y_reg[tr_idx], y_reg[va_idx], y_reg[te_idx]
TUNING_LOG, REG_FITTED, REG_BEST_PARAMS = [], {}, {}
for kind in ["dummy_mean", "ridge", "random_forest", "hist_gradient_boosting"]:
    M = MATS[FAMILY[kind]]
    best = None
    for params in REG_GRIDS.get(kind, [{}]):
        t1 = time.time()
        est = reg_estimator(kind, **params).fit(M["train"], y_tr)
        val_mae = mae(y_va, est.predict(M["val"]))
        TUNING_LOG.append({"task": "regression", "model": kind, "params": json.dumps(params), "val_MAE": val_mae, "seconds": round(time.time() - t1, 1)})
        if best is None or val_mae < best[0]:
            best = (val_mae, params, est)
    REG_FITTED[kind], REG_BEST_PARAMS[kind] = best[2], best[1]
    print(f"regression {kind:24s} best params {best[1]}  val MAE {best[0]:.4f}")

REG_ROWS, REG_TEST_PRED = [], {}
for kind, est in REG_FITTED.items():
    M = MATS[FAMILY[kind]]
    pv, pt = est.predict(M["val"]), est.predict(M["test"])
    REG_TEST_PRED[kind] = np.clip(pt, 0, 10)
    lo, hi = bootstrap_ci(y_te, pt, mae, n_boot=CONFIG["N_BOOTSTRAP"], seed=SEED)
    REG_ROWS.append({"model": kind, "params": json.dumps(REG_BEST_PARAMS[kind]), **{f"val_{k}": v for k, v in reg_metrics(y_va, pv).items() if k in ("MAE", "RMSE", "R2")},
                     **{f"test_{k}": v for k, v in reg_metrics(y_te, pt).items()}, "test_MAE_ci_low": lo, "test_MAE_ci_high": hi,
                     "test_band_accuracy": float(np.mean(score_to_band(pt, TV) == d_tv["label"].values[te_idx]))})
reg_cmp = pd.DataFrame(REG_ROWS).sort_values("val_MAE").reset_index(drop=True)
display(reg_cmp[["model", "val_MAE", "val_R2", "test_MAE", "test_MAE_ci_low", "test_MAE_ci_high", "test_RMSE", "test_R2", "test_resid_mean", "test_resid_std", "test_band_accuracy"]].round(4))
cand = reg_cmp[reg_cmp.model != "dummy_mean"]
REG_SELECTED = cand.iloc[0]["model"]  # selection uses VALIDATION MAE only
dummy_val = float(reg_cmp.loc[reg_cmp.model == "dummy_mean", "val_MAE"].iloc[0])
assert float(cand.iloc[0]["val_MAE"]) < dummy_val, "No candidate beats the Dummy baseline on validation -- do not export a model."
print(f"selected regressor (by validation MAE): {REG_SELECTED}")

In [ ]:
# ---------------- classification ----------------
cls_ok = lambda ix: ix[~np.isnan(y_cls_all[ix])]
ctr, cva, cte = cls_ok(tr_idx), cls_ok(va_idx), cls_ok(te_idx)
# classification rows are a subset of the regression rows; map them to positions inside each split's matrices
pos = lambda full, sub: np.searchsorted(full, sub)
ptr, pva, pte = pos(tr_idx, ctr), pos(va_idx, cva), pos(te_idx, cte)
yc_tr, yc_va, yc_te = y_cls_all[ctr].astype(int), y_cls_all[cva].astype(int), y_cls_all[cte].astype(int)
CLS_LABELS = list(range(len(CLASSES)))
CLS_GRIDS = {"hist_gradient_boosting": [{"max_leaf_nodes": ml} for ml in (31, 63)]}


def cls_estimator(kind, **kw):
    return {"dummy_prior": lambda: DummyClassifier(strategy="prior"),
            "logistic_regression": lambda: LogisticRegression(C=1.0, class_weight="balanced", max_iter=300, random_state=SEED),
            "random_forest": lambda: RandomForestClassifier(n_estimators=150, min_samples_leaf=3, max_features="sqrt", max_samples=0.5, class_weight="balanced_subsample", n_jobs=N_JOBS, random_state=SEED),
            "hist_gradient_boosting": lambda: HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, max_leaf_nodes=kw.get("max_leaf_nodes", 31),
                                                                             l2_regularization=1.0, early_stopping=False, random_state=SEED)}[kind]()


CFAMILY = {"dummy_prior": "linear", "logistic_regression": "linear", "random_forest": "tree", "hist_gradient_boosting": "tree"}


def fit_cls(est, kind, X, y):
    if kind == "hist_gradient_boosting":  # class-balanced weights (HGB has no universal class_weight arg across sklearn versions)
        return est.fit(X, y, sample_weight=compute_sample_weight("balanced", y))
    return est.fit(X, y)


CLS_FITTED, CLS_BEST_PARAMS = {}, {}
for kind in ["dummy_prior", "logistic_regression", "random_forest", "hist_gradient_boosting"]:
    M = MATS[CFAMILY[kind]]
    best = None
    for params in CLS_GRIDS.get(kind, [{}]):
        t1 = time.time()
        est = fit_cls(cls_estimator(kind, **params), kind, M["train"][ptr], yc_tr)
        f1v = f1_score(yc_va, est.predict(M["val"][pva]), average="macro", labels=CLS_LABELS, zero_division=0)
        TUNING_LOG.append({"task": "classification", "model": kind, "params": json.dumps(params), "val_f1_macro": f1v, "seconds": round(time.time() - t1, 1)})
        if best is None or f1v > best[0]:
            best = (f1v, params, est)
    CLS_FITTED[kind], CLS_BEST_PARAMS[kind] = best[2], best[1]
    print(f"classification {kind:22s} best params {best[1]}  val macro-F1 {best[0]:.4f}")

CLS_ROWS, CLS_TEST_PROBA = [], {}
for kind, est in CLS_FITTED.items():
    M = MATS[CFAMILY[kind]]
    prob_t = est.predict_proba(M["test"][pte])
    if prob_t.shape[1] != len(CLASSES):  # a model may have seen fewer classes -> align columns
        full = np.zeros((len(pte), len(CLASSES))); full[:, est.classes_.astype(int)] = prob_t; prob_t = full
    CLS_TEST_PROBA[kind] = prob_t
    pred_t = prob_t.argmax(1)
    pred_v = est.predict(M["val"][pva])
    lo, hi = bootstrap_ci(yc_te, pred_t, lambda a, b: f1_score(a, b, average="macro", labels=CLS_LABELS, zero_division=0), n_boot=CONFIG["N_BOOTSTRAP"], seed=SEED, stratify=yc_te)
    try:
        auc = roc_auc_score(label_binarize(yc_te, classes=CLS_LABELS), prob_t, average="macro")
        pr_auc = average_precision_score(label_binarize(yc_te, classes=CLS_LABELS), prob_t, average="macro")
    except ValueError:
        auc = pr_auc = float("nan")
    CLS_ROWS.append({"model": kind, "params": json.dumps(CLS_BEST_PARAMS[kind]), "val_f1_macro": float(f1_score(yc_va, pred_v, average="macro", labels=CLS_LABELS, zero_division=0)),
                     **{f"test_{k}": v for k, v in cls_metrics(yc_te, pred_t, CLS_LABELS).items()}, "test_f1_macro_ci_low": lo, "test_f1_macro_ci_high": hi,
                     "test_roc_auc_ovr_macro": float(auc), "test_pr_auc_ovr_macro": float(pr_auc)})
cls_cmp = pd.DataFrame(CLS_ROWS).sort_values("val_f1_macro", ascending=False).reset_index(drop=True)
display(cls_cmp[["model", "val_f1_macro", "test_accuracy", "test_balanced_accuracy", "test_precision_macro", "test_recall_macro", "test_f1_macro", "test_f1_macro_ci_low", "test_f1_macro_ci_high",
                 "test_f1_weighted", "test_roc_auc_ovr_macro", "test_pr_auc_ovr_macro"]].round(4))
ccand = cls_cmp[cls_cmp.model != "dummy_prior"]
CLS_SELECTED = ccand.iloc[0]["model"]  # selection uses VALIDATION macro-F1 only
assert float(ccand.iloc[0]["val_f1_macro"]) > float(cls_cmp.loc[cls_cmp.model == "dummy_prior", "val_f1_macro"].iloc[0]), "No classifier beats the Dummy baseline on validation."
print(f"selected classifier (by validation macro-F1): {CLS_SELECTED}")
sel_report = pd.DataFrame(classification_report(yc_te, CLS_TEST_PROBA[CLS_SELECTED].argmax(1), labels=CLS_LABELS, target_names=CLASSES, output_dict=True, zero_division=0)).T
display(sel_report.round(4))
print(f"\ntuning log rows: {len(TUNING_LOG)}  (every configuration evaluated is recorded in tuning_log.csv)")

## 14. Final models and held-out evaluation
The selected regressor and classifier are **refit on train + validation** (more, and more recent, data) and scored **once** on the untouched chronological test split. These two pipelines are what gets exported.

In [ ]:
spec_head = FEATURE_SETS[HEADLINE_SET]
HEAD_COLS = feature_columns(spec_head)
trval = np.concatenate([tr_idx, va_idx])
ctv = cls_ok(trval)

final_reg = Pipeline([("prep", make_prep(spec_head, FAMILY[REG_SELECTED])), ("model", reg_estimator(REG_SELECTED, **REG_BEST_PARAMS[REG_SELECTED]))])
final_reg.fit(Xall.iloc[trval][HEAD_COLS], y_reg[trval])
final_cls = Pipeline([("prep", make_prep(spec_head, CFAMILY[CLS_SELECTED])), ("model", cls_estimator(CLS_SELECTED, **CLS_BEST_PARAMS[CLS_SELECTED]))])
fit_kw = {"model__sample_weight": compute_sample_weight("balanced", y_cls_all[ctv].astype(int))} if CLS_SELECTED == "hist_gradient_boosting" else {}
final_cls.fit(Xall.iloc[ctv][HEAD_COLS], y_cls_all[ctv].astype(int), **fit_kw)

pt_final = np.clip(final_reg.predict(Xall.iloc[te_idx][HEAD_COLS]), 0, 10)
FINAL_REG = reg_metrics(y_te, pt_final)
FINAL_REG["MAE_ci95"] = bootstrap_ci(y_te, pt_final, mae, n_boot=CONFIG["N_BOOTSTRAP"], seed=SEED)
FINAL_REG["band_accuracy"] = float(np.mean(score_to_band(pt_final, TV) == d_tv["label"].values[te_idx]))
prob_final = final_cls.predict_proba(Xall.iloc[cte][HEAD_COLS])
pc_final = prob_final.argmax(1)
FINAL_CLS = cls_metrics(yc_te, pc_final, CLS_LABELS)
FINAL_CLS["f1_macro_ci95"] = bootstrap_ci(yc_te, pc_final, lambda a, b: f1_score(a, b, average="macro", labels=CLS_LABELS, zero_division=0), n_boot=CONFIG["N_BOOTSTRAP"], seed=SEED, stratify=yc_te)
Y_BIN = label_binarize(yc_te, classes=CLS_LABELS)
FINAL_CLS["roc_auc_ovr_macro"] = float(roc_auc_score(Y_BIN, prob_final, average="macro"))
FINAL_CLS["pr_auc_ovr_macro"] = float(average_precision_score(Y_BIN, prob_final, average="macro"))
DUMMY_MAE = float(reg_cmp.loc[reg_cmp.model == "dummy_mean", "test_MAE"].iloc[0])
DUMMY_F1 = float(cls_cmp.loc[cls_cmp.model == "dummy_prior", "test_f1_macro"].iloc[0])
print(f"FINAL regressor  ({REG_SELECTED}) TEST: MAE {FINAL_REG['MAE']:.3f} (95% CI {FINAL_REG['MAE_ci95'][0]:.3f}-{FINAL_REG['MAE_ci95'][1]:.3f}), RMSE {FINAL_REG['RMSE']:.3f}, R2 {FINAL_REG['R2']:.3f}")
print(f"FINAL classifier ({CLS_SELECTED}) TEST: macro-F1 {FINAL_CLS['f1_macro']:.3f} (95% CI {FINAL_CLS['f1_macro_ci95'][0]:.3f}-{FINAL_CLS['f1_macro_ci95'][1]:.3f}), balanced acc {FINAL_CLS['balanced_accuracy']:.3f}, accuracy {FINAL_CLS['accuracy']:.3f}")
print(f"Dummy baselines  TEST: MAE {DUMMY_MAE:.3f}, macro-F1 {DUMMY_F1:.3f}")

## 15. Ablation, repeated-split variability and performance over time
* **Feature-set ablation (RQ3):** same models/hyper-parameters on `text_only`, `core`, `core_plus_enrichment` (chronological split).
* **Chronological vs. random split:** `N_REPEATED_SPLITS` stratified random splits (mean ± sd) vs the single chronological result → quantifies how optimistic random splitting is.
* **Expanding-window by year (Ridge only, for cost):** train on all earlier years, test on each of the last three calendar years.

In [ ]:
ABL_ROWS = []
for set_name in FEATURE_SETS:
    for kind in ("ridge", "hist_gradient_boosting"):
        fam = FAMILY[kind]
        if set_name == HEADLINE_SET:
            pte_ = REG_FITTED[kind].predict(MATS[fam]["test"])
        else:
            prep_, Mtr_, oth_ = build_matrices(set_name, fam, tr_idx, {"test": te_idx})
            pte_ = reg_estimator(kind, **REG_BEST_PARAMS[kind]).fit(Mtr_, y_tr).predict(oth_["test"])
        m = reg_metrics(y_te, pte_)
        lo, hi = bootstrap_ci(y_te, pte_, mae, n_boot=CONFIG["N_BOOTSTRAP"], seed=SEED)
        ABL_ROWS.append({"feature_set": set_name, "model": kind, "n_input_columns": len(feature_columns(FEATURE_SETS[set_name])), "test_MAE": m["MAE"], "test_R2": m["R2"],
                         "test_MAE_ci_low": lo, "test_MAE_ci_high": hi})
ablation_df = pd.DataFrame(ABL_ROWS)
display(ablation_df.round(4))

REP_ROWS = []
for i in range(CONFIG["N_REPEATED_SPLITS"]):
    a, b, c = random_split(SEED + i)
    for kind in ("ridge", "hist_gradient_boosting"):
        pipe = Pipeline([("prep", make_prep(spec_head, FAMILY[kind])), ("model", reg_estimator(kind, **REG_BEST_PARAMS[kind]))]).fit(Xall.iloc[a][HEAD_COLS], y_reg[a])
        m = reg_metrics(y_reg[c], pipe.predict(Xall.iloc[c][HEAD_COLS]))
        REP_ROWS.append({"repeat": i, "seed": SEED + i, "model": kind, "test_MAE": m["MAE"], "test_R2": m["R2"]})
rep_df = pd.DataFrame(REP_ROWS)
display(rep_df.groupby("model")[["test_MAE", "test_R2"]].agg(["mean", "std"]).round(4))
chrono_vs_random = pd.DataFrame([{"model": k, "protocol": "chronological (single test split)", "MAE": reg_cmp.loc[reg_cmp.model == k, "test_MAE"].iloc[0]} for k in ("ridge", "hist_gradient_boosting")]
                                + [{"model": k, "protocol": f"random stratified (mean of {CONFIG['N_REPEATED_SPLITS']})", "MAE": g["test_MAE"].mean()} for k, g in rep_df.groupby("model")])

YEAR_ROWS = []
for yr in sorted(d_tv["pub_year"].unique())[-3:]:
    a, c = np.where(d_tv["pub_year"].values < yr)[0], np.where(d_tv["pub_year"].values == yr)[0]
    if len(a) < 2000 or len(c) < 200:
        continue
    pipe = Pipeline([("prep", make_prep(spec_head, "linear")), ("model", reg_estimator("ridge", **REG_BEST_PARAMS["ridge"]))]).fit(Xall.iloc[a][HEAD_COLS], y_reg[a])
    m = reg_metrics(y_reg[c], pipe.predict(Xall.iloc[c][HEAD_COLS]))
    YEAR_ROWS.append({"test_year": int(yr), "n_train": len(a), "n_test": len(c), "MAE": m["MAE"], "R2": m["R2"]})
year_df = pd.DataFrame(YEAR_ROWS)
display(year_df.round(4))

## 16. Required visualisations (Experiment B)
ROC / precision-recall curves are drawn **one-vs-rest per class** from the classifier's predicted probabilities (valid for a multi-class probabilistic classifier). Regression plots use the final regressor on the chronological test split.

In [ ]:
rs = np.random.default_rng(SEED).choice(len(y_te), size=min(len(y_te), CONFIG["PLOT_SAMPLE_ROWS"]), replace=False)
# B1 actual vs predicted
fig, ax = plt.subplots(figsize=(6, 5.5))
hb = ax.hexbin(y_te[rs], pt_final[rs], gridsize=45, extent=(0, 10, 0, 10), cmap="viridis", bins="log", mincnt=1)
ax.plot([0, 10], [0, 10], "r--", lw=1, label="perfect prediction"); fig.colorbar(hb, label="CVE count (log)")
ax.set(xlabel=f"Official {TV_NAME} base score", ylabel=f"Predicted score ({REG_SELECTED})", title=f"Exp B: actual vs predicted (test, metadata-only)\nMAE {FINAL_REG['MAE']:.2f}, R2 {FINAL_REG['R2']:.2f}"); ax.legend()
savefig("B1_actual_vs_predicted")
# B2 residuals
resid = y_te - pt_final
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].hist(resid, bins=60, color="#8a5a9e"); axes[0].axvline(0, color="r", lw=1)
axes[0].set(title=f"Exp B residual distribution (actual - predicted)\nmean {resid.mean():.2f}, sd {resid.std():.2f}, p5/p95 {np.percentile(resid,5):.1f}/{np.percentile(resid,95):.1f}", xlabel="residual (score points)", ylabel="CVE count")
hb = axes[1].hexbin(pt_final[rs], resid[rs], gridsize=40, cmap="magma", bins="log", mincnt=1); axes[1].axhline(0, color="c", lw=1); fig.colorbar(hb, ax=axes[1], label="count (log)")
axes[1].set(title="Exp B residual vs predicted score", xlabel="predicted score", ylabel="residual (score points)")
savefig("B2_residuals")
# B3 model comparison (test, with CIs)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
r_ = reg_cmp.sort_values("test_MAE", ascending=False)
axes[0].barh(r_["model"], r_["test_MAE"], xerr=[r_["test_MAE"] - r_["test_MAE_ci_low"], r_["test_MAE_ci_high"] - r_["test_MAE"]], color="#3b6ea5", capsize=3)
axes[0].set(title="Regression: test MAE by model (fit on train; 95% bootstrap CI)", xlabel="MAE (score points; lower is better)")
c_ = cls_cmp.sort_values("test_f1_macro")
axes[1].barh(c_["model"], c_["test_f1_macro"], xerr=[c_["test_f1_macro"] - c_["test_f1_macro_ci_low"], c_["test_f1_macro_ci_high"] - c_["test_f1_macro"]], color="#5b9a5b", capsize=3)
axes[1].set(title="Classification: test macro-F1 by model (95% bootstrap CI)", xlabel="macro-F1 (higher is better)")
savefig("B3_model_comparison")
# B4 confusion matrix
cmx = confusion_matrix(yc_te, pc_final, labels=CLS_LABELS)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.heatmap(cmx, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0]); axes[0].set(title=f"Confusion matrix (counts), {CLS_SELECTED}, test", xlabel="predicted", ylabel="official")
sns.heatmap(cmx / cmx.sum(1, keepdims=True), annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1]); axes[1].set(title="Row-normalised (recall per official class)", xlabel="predicted", ylabel="official")
savefig("B4_confusion_matrix")
# B5/B6 one-vs-rest ROC and PR curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for i, cname in enumerate(CLASSES):
    if Y_BIN[:, i].sum() == 0:
        continue
    fpr, tpr, _ = roc_curve(Y_BIN[:, i], prob_final[:, i]); axes[0].plot(fpr, tpr, label=f"{cname} (AUC {roc_auc_score(Y_BIN[:, i], prob_final[:, i]):.2f})")
    pr_, rc_, _ = precision_recall_curve(Y_BIN[:, i], prob_final[:, i]); axes[1].plot(rc_, pr_, label=f"{cname} (AP {average_precision_score(Y_BIN[:, i], prob_final[:, i]):.2f}; prevalence {Y_BIN[:, i].mean():.2f})")
axes[0].plot([0, 1], [0, 1], "k:", lw=1); axes[0].set(title="One-vs-rest ROC (test)", xlabel="false-positive rate", ylabel="true-positive rate"); axes[0].legend(fontsize=8)
axes[1].set(title="One-vs-rest precision-recall (test)", xlabel="recall", ylabel="precision", ylim=(0, 1.02)); axes[1].legend(fontsize=8)
savefig("B5_B6_roc_and_precision_recall")
# B7 ablation + B8 chrono-vs-random
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
sns.barplot(data=ablation_df, x="feature_set", y="test_MAE", hue="model", ax=axes[0]); axes[0].set(title="Feature-set ablation (chronological test MAE)", xlabel="", ylabel="MAE (score points)")
sns.barplot(data=chrono_vs_random, x="model", y="MAE", hue="protocol", ax=axes[1])
axes[1].set(title="Chronological vs random-split MAE", xlabel="", ylabel="MAE (score points)"); axes[1].legend(fontsize=8)
savefig("B7_B8_ablation_and_split_protocol")
if len(year_df):
    fig, ax = plt.subplots(figsize=(6, 4)); ax.bar(year_df["test_year"].astype(str), year_df["MAE"], color="#d08a3b")
    for i, r in year_df.iterrows():
        ax.text(i, r["MAE"], f"n={r['n_test']:,}", ha="center", va="bottom", fontsize=8)
    ax.set(title="Expanding-window MAE by test year (Ridge, metadata-only)", xlabel="test year (trained on all earlier years)", ylabel="MAE (score points)")
    savefig("B9_performance_by_year")

## 17. Error and subgroup analysis
Subgroups with fewer than 30 test rows are flagged (grey bars) and should not be interpreted. Scorer = who produced the reported score (`cvss_*_source`): NVD analysts vs CNAs/others.

In [ ]:
tst = d_tv.iloc[te_idx].copy()
tst["pred"], tst["abs_err"] = pt_final, np.abs(y_te - pt_final)
tst["band_ok"] = score_to_band(pt_final, TV) == tst["label"].values
tst["scorer"] = np.where(tst[f"cvss_{TV}_source"].astype(str).str.lower() == "nvd@nist.gov", "NVD", "CNA/other")
tst["month"] = tst["published_date"].dt.to_period("M").astype(str)
tst["text_novelty"] = np.where(tst["text_seen_in_train"], "text seen in train", "novel text")
SUB_ROWS = []
for dim, col, top in [("scorer", "scorer", None), ("publication month", "month", None), ("text novelty (near-duplicate check)", "text_novelty", None),
                      ("primary CWE (top 8)", "primary_cwe_id", 8), ("reporting CNA (top 8)", "source_identifier", 8), ("official class", "label", None)]:
    g = tst[col].fillna("(missing)")
    keep = g.value_counts().head(top).index if top else g.unique()
    for val in sorted(keep, key=str):
        s = tst[g == val]
        SUB_ROWS.append({"dimension": dim, "group": str(val), "n": len(s), "MAE": s["abs_err"].mean(), "R2": r2_score(s["y"], s["pred"]) if (len(s) > 2 and s["y"].nunique() > 1 and dim != "official class") else np.nan,  # R2 inside a slice of the target itself is meaningless
                         "band_accuracy": s["band_ok"].mean(), "low_n_flag": len(s) < 30})
subgroup_df = pd.DataFrame(SUB_ROWS)
display(subgroup_df.round(3))
dims = subgroup_df["dimension"].unique()
fig, axes = plt.subplots(2, 3, figsize=(17, 8.5))
for ax, dim in zip(axes.ravel(), dims):
    s = subgroup_df[subgroup_df.dimension == dim].reset_index(drop=True)
    ax.bar(s["group"].str.slice(0, 16), s["MAE"], color=np.where(s["low_n_flag"], "#bbbbbb", "#3b6ea5"))
    for i, r in s.iterrows():
        ax.text(i, r["MAE"], f"n={r['n']}", ha="center", va="bottom", fontsize=7)
    ax.set(title=f"Test MAE by {dim}", ylabel="MAE (score points)"); ax.tick_params(axis="x", rotation=45)
    ax.axhline(FINAL_REG["MAE"], color="r", ls="--", lw=1)
savefig("B10_subgroup_performance")
worst = tst.sort_values("abs_err", ascending=False).head(8)[["cve_id", "y", "pred", "label", "scorer", "primary_cwe_id", "description_en"]].copy()
worst["description_en"] = worst["description_en"].str.slice(0, 110)
print("Largest test errors (regression):"); display(worst.round(2))

## 18. Model interpretation
Permutation importance (MAE increase when a column is shuffled) on a validation sample using the **train-only** selected regressor — SHAP is intentionally not required. Ridge coefficients show which description terms push the predicted score up/down. Importance ≠ causation: terms like "remote" or "SQL injection" mirror how scorers *describe* attack vectors.

In [ ]:
head_pipe = Pipeline([("prep", MATS[FAMILY[REG_SELECTED]]["prep"]), ("model", REG_FITTED[REG_SELECTED])])
vs = np.random.default_rng(SEED).choice(len(va_idx), size=min(len(va_idx), CONFIG["PERM_IMPORTANCE_ROWS"]), replace=False)
Xv, yv = Xall.iloc[va_idx[vs]][HEAD_COLS], y_va[vs]
pi = permutation_importance(head_pipe, Xv, yv, scoring="neg_mean_absolute_error", n_repeats=3, random_state=SEED, n_jobs=1)
pi_df = pd.DataFrame({"feature": HEAD_COLS, "mae_increase": pi.importances_mean, "sd": pi.importances_std}).sort_values("mae_increase", ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
top = pi_df.head(15).iloc[::-1]
axes[0].barh(top["feature"], top["mae_increase"], xerr=top["sd"], color="#3b6ea5", capsize=2); axes[0].set(title=f"Permutation importance ({REG_SELECTED}, validation sample n={len(vs):,})", xlabel="increase in MAE when shuffled (score points)")
rid = REG_FITTED["ridge"]; names = MATS["linear"]["prep"].get_feature_names_out()
coef = pd.Series(rid.coef_, index=names); terms = coef[coef.index.str.startswith("text__")]; terms.index = terms.index.str.replace("text__", "", regex=False)
tt = pd.concat([terms.nsmallest(12), terms.nlargest(12)]).sort_values()
axes[1].barh(tt.index, tt.values, color=np.where(tt.values > 0, "#c0504d", "#4f81bd")); axes[1].set(title="Ridge: description terms with the largest coefficients\n(red: raise predicted score, blue: lower it)", xlabel="coefficient (score points per unit TF-IDF)")
savefig("B11_feature_importance")
display(pi_df.head(10).round(4))

## 19. Model limitations and intended use

**Intended use (research only):** triage-time *estimate* of CVSS severity for a freshly published CVE whose official score does not exist yet, to study whether text/CNA/CWE metadata carry severity signal.

**Not intended / prohibited interpretations**
* Not a replacement for, or override of, an official CVSS score. Where a CVSS vector exists, the **deterministic calculator is exact and authoritative** (Experiment A shows ML only approximates it).
* Not an exploitability or environmental-risk estimate. Severity ≠ exploit likelihood ≠ business risk (see notebooks 02 and 03).
* Not validated outside NVD-published CVEs of the training period; no claim about vendor-specific advisories, cloud findings, or scanner output that is not a CVE.
* Predicted scores/classes carry sizeable error (see test CIs and subgroup table); never present them as official scores.

**Known limitations:** heterogeneous scorers (NVD vs. many CNAs) put a ceiling on learnability — two analysts can legitimately score the same description differently; the snapshot is the *current* NVD state, so features such as CWE may post-date publication; descriptions are partly templated (near-duplicates inflate random-split scores); CVSS v4.0 has too few rows and no calculator here; label noise from CNA mis-scoring is unmeasured; confidence intervals cover test-sampling variability only, not dataset-shift or scorer-drift.

### When is deterministic CVSS calculation preferable to ML?
1. **Whenever a base vector exists** — the formula is exact, free, auditable and reproducible; Experiment A shows the best ML model still leaves non-zero error, and the calculator error is exactly zero.
2. **Whenever the number must be explained** (reports, audits, SLAs): a vector → score derivation is transparent; a tree ensemble is not.
3. **Whenever inputs may be adversarial or out of distribution** — an ML model degrades silently; a formula rejects an invalid vector.
ML metadata models are only defensible *before* a vector exists, as a clearly labelled estimate, and only if they beat the baseline by a margin that matters operationally.

## 20. Artifact export and reload verification
Everything needed to audit or reload the experiment is written to `OUTPUT_DIR` (git-ignored; never commit models or datasets). The smoke test **reloads the exported joblib files and verifies inference on raw example rows** with the exported input schema.

In [ ]:
MODEL_DIR = OUTPUT_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)
FEATURE_SCHEMA = {
    "schema_name": "offensivex-ml-research/cve-severity-input", "schema_version": "1.0.0",
    "status": "EXPERIMENTAL research artifact -- not for production use",
    "target": {"cvss_version": TV_NAME, "regression": "predicted_cvss_base_score (0.0-10.0)", "classification_classes": CLASSES,
               "offensivex_severity_mapping": {c: CVSS_TO_OFFENSIVEX_SEVERITY[c] for c in CLASSES}},
    "raw_inputs": {
        "description_en": {"dtype": "string", "required": True, "missing_behavior": "REJECT (non-empty English CVE description required)"},
        "source_identifier": {"dtype": "string", "required": False, "missing_behavior": "treated as unknown/infrequent category"},
        "primary_cwe_id": {"dtype": "string", "required": False, "pattern": r"^(CWE-\d+|NVD-CWE-(noinfo|Other))$", "missing_behavior": "treated as unknown/infrequent category"},
    },
    "derived_features": {"computed_by": "add_text_features() (reference implementation: FLAG_PATTERNS below)", "columns": ["desc_length", "desc_words"] + FLAG_COLUMNS, "flag_patterns": FLAG_PATTERNS},
    "model_columns": HEAD_COLS,
    "forbidden_inputs": "any cvss_* / epss_* / kev_* field, vuln_status, last_modified, reference_count, cpe_match_count (leakage controls; see notebook section 9)",
}


def prepare_inputs(raw, schema=FEATURE_SCHEMA):
    """Validate raw rows against the schema and derive model columns. Raises ValueError with an actionable message."""
    need = [c for c, s in schema["raw_inputs"].items() if s["required"]]
    missing = [c for c in need if c not in raw.columns]
    if missing:
        raise ValueError(f"missing required input column(s): {missing}")
    forbidden = [c for c in raw.columns if FORBIDDEN_RE.search(c)]
    if forbidden:
        raise ValueError(f"forbidden (leakage-prone) input column(s) supplied: {forbidden}")
    x = raw.copy()
    for c in schema["raw_inputs"]:
        if c not in x.columns:
            x[c] = None
    if x["description_en"].isna().any() or x["description_en"].astype(str).str.strip().eq("").any():
        raise ValueError("description_en must be non-empty for every row")
    return add_text_features(x.assign(description_en=x["description_en"].astype(str)))[schema["model_columns"]]


joblib.dump(final_reg, MODEL_DIR / "severity_regressor.joblib", compress=3)
joblib.dump(final_cls, MODEL_DIR / "severity_classifier.joblib", compress=3)
write_json(OUTPUT_DIR / "feature_schema.json", FEATURE_SCHEMA)
write_json(OUTPUT_DIR / "target_definition.json", {
    "regression_target": f"reported {TV_NAME} base score (float 0-10)", "classification_target": f"{TV_NAME} qualitative band derived from the score",
    "band_thresholds": "NONE 0.0 | LOW 0.1-3.9 | MEDIUM 4.0-6.9 | HIGH 7.0-8.9 | CRITICAL 9.0-10.0 (v2: LOW <4, MEDIUM <7, HIGH >=7)",
    "label_mapping_index_to_class": dict(enumerate(CLASSES)), "classes_excluded_from_classification": dropped_classes,
    "offensivex_severity_mapping": CVSS_TO_OFFENSIVEX_SEVERITY, "split": "chronological 70/15/15 by published_date (headline)"})
write_json(OUTPUT_DIR / "hyperparameters.json", {"regression": {"selected": REG_SELECTED, "params_by_model": REG_BEST_PARAMS}, "classification": {"selected": CLS_SELECTED, "params_by_model": CLS_BEST_PARAMS},
                                                 "tfidf_max_features": CONFIG["TFIDF_MAX_FEATURES"], "svd_components": CONFIG["SVD_COMPONENTS"], "seed": SEED})
METRICS = {"fast_dev": CONFIG["FAST_DEV"], "target_version": TV_NAME, "n_rows": {"analysis_table": len(d_tv), "train": len(tr_idx), "validation": len(va_idx), "test": len(te_idx)},
           "experiment_A_reconstruction": {**RECON_SUMMARY, "interpretation": "benchmark of re-deriving the deterministic formula; NOT predictive capability"},
           "experiment_B_regression_test": {"selected_model": REG_SELECTED, **FINAL_REG, "dummy_MAE": DUMMY_MAE},
           "experiment_B_classification_test": {"selected_model": CLS_SELECTED, **FINAL_CLS, "dummy_f1_macro": DUMMY_F1, "per_class": sel_report.round(4).to_dict()},
           "split_protocol_comparison": chrono_vs_random.to_dict(orient="records")}
write_json(OUTPUT_DIR / "metrics.json", METRICS)
export_common(tables={"model_comparison_regression": reg_cmp, "model_comparison_classification": cls_cmp, "reconstruction_benchmark": recon_df, "ablation_feature_sets": ablation_df,
                      "repeated_random_splits": rep_df, "performance_by_year": year_df, "subgroup_metrics": subgroup_df, "tuning_log": pd.DataFrame(TUNING_LOG),
                      "leakage_table": leakage_table, "version_audit": version_audit, "split_table": split_table, "permutation_importance": pi_df},
              extra_config={"fast_dev": CONFIG["FAST_DEV"], "headline_feature_set": HEADLINE_SET})

# ---- reload verification (fresh objects from disk, raw example rows, schema-validated) ----
reg_loaded, cls_loaded = joblib.load(MODEL_DIR / "severity_regressor.joblib"), joblib.load(MODEL_DIR / "severity_classifier.joblib")
raw_demo = d_tv.iloc[te_idx].head(5)[["description_en", "source_identifier", "primary_cwe_id"]].reset_index(drop=True)
x_demo = prepare_inputs(raw_demo)
p_reg, p_cls = np.clip(reg_loaded.predict(x_demo), 0, 10), cls_loaded.predict_proba(x_demo)
assert np.allclose(p_reg, np.clip(final_reg.predict(Xall.iloc[te_idx].head(5)[HEAD_COLS]), 0, 10)), "reloaded regressor disagrees with in-memory model"
assert np.allclose(p_cls, final_cls.predict_proba(Xall.iloc[te_idx].head(5)[HEAD_COLS])), "reloaded classifier disagrees with in-memory model"
assert np.allclose(p_cls.sum(1), 1.0) and ((p_reg >= 0) & (p_reg <= 10)).all()
# optional-field behaviour and rejection behaviour
_ = reg_loaded.predict(prepare_inputs(raw_demo.drop(columns=["source_identifier", "primary_cwe_id"]).head(2)))
for bad_input, why in [(raw_demo.assign(description_en="").head(1), "empty description"), (raw_demo.assign(cvss_v31_base_score=9.8).head(1), "leakage column supplied")]:
    try:
        prepare_inputs(bad_input)
    except ValueError:
        continue
    raise AssertionError(f"input with {why} was NOT rejected")
demo = pd.DataFrame({"cve_id": d_tv.iloc[te_idx].head(5)["cve_id"].values, "official_score": y_te[:5], "predicted_score": p_reg.round(2),
                     "predicted_class": [CLASSES[i] for i in p_cls.argmax(1)], "official_class": d_tv.iloc[te_idx].head(5)["label"].values})
display(demo)
print("SMOKE TEST PASSED: models reloaded from disk, inference matches in-memory models, schema validation accepts valid and rejects invalid input.")
SMOKE_OK = True

## 21. Reproducibility checklist and conclusions
Conclusions are generated from this run's numbers — nothing below is hand-written.

In [ ]:
repro = pd.DataFrame([
    ("random seed", CONFIG["SEED"]), ("dataset fingerprint (sha256)", DATA_META["dataset_fingerprint_sha256"][:24] + "..."),
    ("data source", DATA_META["source_kind"]), ("rows analysed / train / val / test", f"{len(d_tv):,} / {len(tr_idx):,} / {len(va_idx):,} / {len(te_idx):,}"),
    ("split protocol (headline)", "chronological 70/15/15"), ("python / sklearn / pandas", f"{DEPENDENCY_VERSIONS['python']} / {DEPENDENCY_VERSIONS['sklearn']} / {DEPENDENCY_VERSIONS['pandas']}"),
    ("FAST_DEV (must be False to report)", CONFIG["FAST_DEV"]), ("plots saved", len(SAVED_PLOTS)), ("reload smoke test", SMOKE_OK), ("output dir", str(OUTPUT_DIR))], columns=["item", "value"])
display(repro)

c_gain = (1 - FINAL_REG["MAE"] / DUMMY_MAE) * 100
rnd = chrono_vs_random.pivot(index="model", columns="protocol", values="MAE")
enr = ablation_df.pivot(index="model", columns="feature_set", values="test_MAE")
chrono_col = [c for c in rnd.columns if c.startswith("chronological")][0]
rand_col = [c for c in rnd.columns if c.startswith("random")][0]
lines = [
    f"* **Experiment A (reconstruction).** Best ML model ({RECON_SUMMARY['best_model']}) re-derives the {TV_NAME} score from its base metrics with MAE {RECON_SUMMARY['best_MAE']:.3f} / R2 {RECON_SUMMARY['best_R2']:.4f}; the official calculator reproduces the reported score {'exactly' if RECON_SUMMARY['calculator_exact'] else 'with the residuals shown above'}. This is a formula-learning benchmark, **not** evidence that ML can assess severity better than CVSS.",
    f"* **Experiment B (metadata only, chronological test).** Regressor `{REG_SELECTED}`: MAE {FINAL_REG['MAE']:.2f} (95% CI {FINAL_REG['MAE_ci95'][0]:.2f}-{FINAL_REG['MAE_ci95'][1]:.2f}) vs mean-baseline {DUMMY_MAE:.2f} ({c_gain:.0f}% lower), R2 {FINAL_REG['R2']:.2f}; band accuracy {FINAL_REG['band_accuracy']:.2f}. Classifier `{CLS_SELECTED}`: macro-F1 {FINAL_CLS['f1_macro']:.2f} (95% CI {FINAL_CLS['f1_macro_ci95'][0]:.2f}-{FINAL_CLS['f1_macro_ci95'][1]:.2f}) vs {DUMMY_F1:.2f} for the majority-class baseline.",
    f"* **Gap to the reconstruction ceiling:** metadata-only MAE {FINAL_REG['MAE']:.2f} vs {RECON_SUMMARY['best_MAE']:.2f} when the metrics are known — most of the severity information is *in the analyst's metric choices*, not in the free-text metadata.",
    f"* **Split protocol:** MAE random-split (mean of {CONFIG['N_REPEATED_SPLITS']}) vs chronological — " + "; ".join(f"{m}: {rnd.loc[m, rand_col]:.3f} vs {rnd.loc[m, chrono_col]:.3f}" for m in rnd.index) + " (a lower random-split number means random splitting is optimistic).",
    "* **Post-publication enrichment (RQ3):** test MAE core -> core_plus_enrichment — " + "; ".join(f"{m}: {enr.loc[m, 'core']:.3f} -> {enr.loc[m, 'core_plus_enrichment']:.3f}" for m in enr.index) + ".",
    f"* **Imbalance:** rarest class '{CLASSES[0]}' F1 = {float(sel_report.loc[CLASSES[0], 'f1-score']):.2f} — minority-class performance is much weaker than the headline macro average suggests.",
    "* **Decision for OffensiveX:** keep deterministic CVSS calculation authoritative; this model is at most a clearly-labelled triage *estimate* for unscored CVEs, and only if a stakeholder decides the observed error is acceptable (ADR-009 still applies).",
]
display(Markdown("\n".join(lines)))

report = f"""# Experiment report — CVE severity prediction ({TV_NAME})
*Generated {datetime.now(timezone.utc).isoformat(timespec='seconds')} by notebooks/01_cve_severity_prediction.ipynb. fast_dev={CONFIG['FAST_DEV']}.*

Status: EXPERIMENTAL research artifact. Not production-ready; see model_card.md.

## Data
{DATA_META['source_description']} — fingerprint `{DATA_META['dataset_fingerprint_sha256']}`; {len(d_tv):,} analysed rows after validation (see exclusion_ledger.csv).

## Conclusions
{chr(10).join(lines)}

## Files
metrics.json, model_comparison_*.csv, reconstruction_benchmark.csv, ablation_feature_sets.csv, subgroup_metrics.csv, tuning_log.csv, feature_schema.json, target_definition.json, hyperparameters.json, models/*.joblib, plots/*.png
"""
(OUTPUT_DIR / "experiment_report.md").write_text(report, encoding="utf-8")
card = f"""# Model card — CVE severity estimators (research prototype)
**Status:** EXPERIMENTAL. fast_dev={CONFIG['FAST_DEV']}. Trained {datetime.now(timezone.utc).date()} on `{DATA_META['dataset_fingerprint_sha256'][:16]}…`.

**Models:** `{REG_SELECTED}` regressor (predicts {TV_NAME} base score) and `{CLS_SELECTED}` classifier (predicts {', '.join(CLASSES)}); sklearn {DEPENDENCY_VERSIONS['sklearn']}, python {DEPENDENCY_VERSIONS['python']}. Reload with the same major versions (`dependency_versions.json`).

**Intended use:** research on whether CVE text/CNA/CWE metadata carry severity signal; offline triage *estimates* for CVEs without an official score.

**Prohibited interpretations:** (1) as an official or authoritative CVSS score; (2) as exploit likelihood or business risk; (3) as a replacement for the deterministic calculator when a vector exists; (4) for any compliance/SLA decision; (5) as validation of OffensiveX's `risk_model_v1`.

**Test performance (chronological holdout):** MAE {FINAL_REG['MAE']:.2f} (CI {FINAL_REG['MAE_ci95'][0]:.2f}-{FINAL_REG['MAE_ci95'][1]:.2f}), R2 {FINAL_REG['R2']:.2f}; macro-F1 {FINAL_CLS['f1_macro']:.2f}. Reconstruction benchmark (not predictive): MAE {RECON_SUMMARY['best_MAE']:.3f}.

**Known biases/limits:** heterogeneous scorers; current-state snapshot; templated descriptions; sparse CVSS v4.0; label noise; no calibration claim for class probabilities; performance on minority class '{CLASSES[0]}' is weak.

**Data/secrets:** no credentials or raw dataset are included in the artifact; `dataset_metadata.json` holds file hashes only.
"""
(OUTPUT_DIR / "model_card.md").write_text(card, encoding="utf-8")
scan_for_secrets(OUTPUT_DIR)
print(f"\nExported {sum(1 for p in OUTPUT_DIR.rglob('*') if p.is_file())} files to {OUTPUT_DIR}")
print("plots:", sorted(SAVED_PLOTS))